# Radar Financeiro — Laboratório Máximo de Performance e Otimização (27 bi — execução segura por estágios)

**Cliente fixo:** `CD_CLI = 468459778`

Este notebook é um laboratório **read-only**, experimental e resiliente.  
Ele mantém:

- a sessão Spark/BBMagic corporativa;
- `GerenciadorLocal`;
- `gerenciador_spark_v2.ipynb`;
- `gerenciador_db2_spark_v2.ipynb`;
- `ConectorDb2Spark`;
- as mesmas fontes físicas do Radar;
- Q3 via Hive/Spark SQL;
- Q1/Q2/Q4/Q5 pelos mecanismos atuais.

## Fontes

| Etapa | Fonte | Mecanismo |
|---|---|---|
| Q1 | `DB2GFP.TRAN_RLZD_INST_PCT` | DB2 / conector corporativo |
| Q2 | `DB2GFP.CT_GRDR_FNCO` | DB2 / conector corporativo |
| Q3 | `DB2DFE.REN_AVLD_PF` | Hive via `spark.sql` |
| Q4 | `DB2D1D.DVS_GRDR_FNCO_PF` | DB2 / conector corporativo |
| Q5 | `DB2GFP.TRAN_RLZD_INST_PCT` | DB2 / conector corporativo |

## Objetivo

Não assumir qual otimização funciona. **Medir.**

Serão testados:

- diferentes formulações SQL;
- formulações que a documentação recomenda;
- formulações que a documentação desaconselha;
- CTEs que podem falhar no wrapper JDBC;
- subqueries;
- `MAX`;
- `ROW_NUMBER`;
- `NOT EXISTS`;
- joins com agregação;
- lookup pontual;
- busca regressiva por competência;
- funções/casts sobre colunas indexadas;
- `BETWEEN` vs limites explícitos;
- projeção mínima vs ampla;
- `fetchsize`;
- `queryTimeout`;
- níveis de isolamento;
- leitura JDBC particionada com o mesmo conector;
- warm/cold;
- persistência e lineage;
- Spark scheduler;
- Hive Q3;
- views temporárias;
- resultado 1×80;
- transporte BBMagic de HTML/payload;
- plano físico;
- metadados e índices reais.

## Segurança operacional

1. Nenhuma escrita em DB2/Hive/HDFS.
2. Nenhum `INSERT`, `UPDATE`, `DELETE`, `MERGE`, `DROP TABLE`, `CREATE TABLE` ou overwrite.
3. O laboratório pode criar apenas **temporary views** e objetos em memória.
4. Consultas potencialmente ruins possuem timeout.
5. Cada teste captura a própria exceção.
6. Uma falha funcional de uma variante não interrompe as variantes seguintes.
7. Se a própria sessão Livy/Spark morrer, nenhuma célula remota consegue continuar até a sessão ser restabelecida; esse é o único limite estrutural da resiliência.

## Importante

Uma query rápida **não é automaticamente candidata à produção**.  
O notebook também mede **equivalência funcional** com a baseline do Radar.

## Política especial para fontes gigantes

O notebook inicia em:

```python
MODO_LAB = "SEGURO"
```

Nesse modo:

- `BAIXO` e `MÉDIO`: executados;
- `ALTO` e `EXTREMO`: permanecem no notebook, mas viram `SKIP`.

Depois de analisar o primeiro relatório, pode-se mudar para:

```python
MODO_LAB = "AGRESSIVO"
```

e, somente se necessário:

```python
MODO_LAB = "EXTREMO"
```

**Motivo:** `TRAN_RLZD_INST_PCT` tem ~27,9 bilhões de linhas e `DVS_GRDR_FNCO_PF` ~1,10 bilhão. Uma função aplicada à coluna errada, um `OR`, um `ROW_NUMBER` histórico ou múltiplas conexões JDBC podem alterar completamente o caminho físico.


## 0. Baseline observada

Referência da execução V2 fornecida anteriormente para o mesmo cliente:

- total: `505.395 s`
- Q1: `5.922 s`
- Q2: `1.009 s`
- Q3: `11.463 s`
- Q4: `93.767 s`
- Q5: `0.636 s`
- resultado oficial: `139.065 s`
- montagem dashboard: `126.357 s`
- Q5 contexto: `83` linhas
- Q5 oficial: `56` linhas
- pares exatos: `2`
- pares de borda: `1`
- efetivas: `51`
- conta: `(3242, 47949)`
- renda: `5941.00`, referência `2026-07-27`
- perfil: `DT_REF=2026-05-01`, macro `3`, micro `6`
- janela: `2026-07-20 .. 2026-08-19`

Os valores são comparação, não gate. Uma base alterada pode legitimamente produzir números diferentes.

## 1. Sessão Spark corporativa — reutilizar se já existir

In [ ]:
from traceback import format_exc
import time

try:
    if globals().get("spark") is not None:
        print("[LAB] Reutilizando objeto local spark já existente:", type(spark))
    else:
        from src.utils.gerenciador_local_v2 import GerenciadorLocal
        gerenciador_local = GerenciadorLocal(
            nome_sessao="radar-laboratorio-maximo",
            exibir_configuracao=False,
            ativar_logs=True,
        )
        spark = gerenciador_local.criar_sessao_spark(db2=True)
        print("[LAB] Sessão Spark criada pelo GerenciadorLocal.")
except Exception as exc:
    print("[LAB][ERRO_SESSAO]", type(exc).__name__, str(exc))
    print(format_exc())

## 2. Utilitários corporativos

In [ ]:
from IPython import get_ipython
import os
from pathlib import Path
import traceback

def run_utilitario(candidatos):
    ip = get_ipython()
    erros = []
    for caminho in candidatos:
        try:
            ip.run_line_magic("run", caminho)
            print("[LAB][UTILITARIO_OK]", caminho)
            return caminho
        except Exception as exc:
            erros.append(f"{caminho}: {type(exc).__name__}: {exc}")
    print("[LAB][UTILITARIO_ERRO]", " | ".join(erros))
    return None

UTIL_SPARK = run_utilitario([
    "./src/utils/gerenciador_spark_v2.ipynb",
    "./gerenciador_spark_v2.ipynb",
])

UTIL_DB2 = run_utilitario([
    "./src/utils/gerenciador_db2_spark_v2.ipynb",
    "./gerenciador_db2_spark_v2.ipynb",
])

## 3. Bootstrap remoto, parâmetros e executor resiliente

In [ ]:
%%spark

import calendar
import datetime
import hashlib
import json
import math
import os
import platform
import re
import statistics
import sys
import time
import traceback
from datetime import timedelta
from decimal import Decimal

from pyspark.sql import Row, Window
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

CD_CLI = 468459778

FONTES = {
    "TRAN": "DB2GFP.TRAN_RLZD_INST_PCT",
    "CICLO": "DB2GFP.CT_GRDR_FNCO",
    "RENDA": "DB2DFE.REN_AVLD_PF",
    "PERFIL": "DB2D1D.DVS_GRDR_FNCO_PF",
}

FETCHSIZE_BASE = 10_000
TIMEOUT_NORMAL = 180
TIMEOUT_AGRESSIVO = 120
REPETICOES_QUERY = 1
REPETICOES_MICRO = 5
DIAS_CONTEXTO_RECONCILIACAO = 5

# Ativado porque este notebook existe especificamente como laboratório.
# -------------------------------------------------------------------------
# CONTROLE DE RISCO
#
# SEGURO      -> BAIXO + MÉDIO
# AGRESSIVO   -> inclui ALTO
# EXTREMO     -> executa tudo, inclusive testes deliberadamente ruins
#
# Comece em SEGURO por causa das fontes:
# TRAN ~27,9 bilhões de linhas
# DVS  ~1,10 bilhão de linhas
# -------------------------------------------------------------------------
MODO_LAB = "SEGURO"

NIVEL_RISCO = {"BAIXO": 0, "MEDIO": 1, "ALTO": 2, "EXTREMO": 3}
LIMITE_RISCO = {
    "SEGURO": NIVEL_RISCO["MEDIO"],
    "AGRESSIVO": NIVEL_RISCO["ALTO"],
    "EXTREMO": NIVEL_RISCO["EXTREMO"],
}[MODO_LAB]

def permitido(risco):
    return NIVEL_RISCO[str(risco).upper()] <= LIMITE_RISCO

def registrar_skip_risco(grupo, teste, estrategia, risco):
    registrar(
        grupo, teste, "SKIP", estrategia=estrategia, inicio=time.perf_counter(),
        detalhe=f"Risco={risco}; MODO_LAB={MODO_LAB}. Teste preservado no notebook, mas não executado neste passe."
    )

EXECUTAR_VARIANTES_ESPERADAS_FALHAR = permitido("ALTO")
EXECUTAR_TESTES_JDBC_PARTICIONADO = permitido("ALTO")
EXECUTAR_TESTES_ISOLAMENTO = permitido("MEDIO")
EXECUTAR_BENCHMARK_TRANSFERENCIA_GRANDE = permitido("MEDIO")

BASELINE_OBSERVADA = {
    "TOTAL_V2_S": 505.395,
    "Q1_S": 5.922,
    "Q2_S": 1.009,
    "Q3_S": 11.463,
    "Q4_S": 93.767,
    "Q5_S": 0.636,
    "RESULTADO_OFICIAL_S": 139.065,
    "DASHBOARD_S": 126.357,
    "Q5_CONTEXTO": 83,
    "Q5_OFICIAL": 56,
    "PARES_EXATOS": 2,
    "PARES_BORDA": 1,
    "EFETIVAS": 51,
    "CONTA": [3242, 47949],
    "RENDA": "5941.00",
    "RENDA_REF": "2026-07-27",
    "PERFIL_REF": "2026-05-01",
    "PERFIL_MACRO": 3,
    "PERFIL_MICRO": 6,
    "JANELA_INI": "2026-07-20",
    "JANELA_FIM": "2026-08-19",
}

RESULTADOS = []
CONTEXTO = {}
PERSISTIDOS = []

def _texto(v, limite=8000):
    try:
        if isinstance(v, (dict, list, tuple)):
            return json.dumps(v, ensure_ascii=False, default=str)[:limite]
        return str(v)[:limite]
    except Exception:
        return repr(v)[:limite]

def registrar(
    grupo,
    teste,
    status="OK",
    estrategia="",
    repeticao=None,
    inicio=None,
    preparacao=None,
    action=None,
    linhas=None,
    colunas=None,
    particoes=None,
    fetchsize=None,
    isolamento=None,
    equivalente=None,
    assinatura=None,
    detalhe="",
):
    total = None if inicio is None else time.perf_counter() - inicio
    item = {
        "ORDEM": len(RESULTADOS) + 1,
        "GRUPO": grupo,
        "TESTE": teste,
        "ESTRATEGIA": estrategia,
        "REPETICAO": repeticao,
        "STATUS": status,
        "TEMPO_PREPARACAO_S": None if preparacao is None else float(preparacao),
        "TEMPO_ACTION_S": None if action is None else float(action),
        "TEMPO_TOTAL_S": None if total is None else float(total),
        "LINHAS": None if linhas is None else int(linhas),
        "COLUNAS": None if colunas is None else int(colunas),
        "PARTICOES": None if particoes is None else int(particoes),
        "FETCHSIZE": fetchsize,
        "ISOLAMENTO": isolamento,
        "EQUIVALENTE_BASELINE": equivalente,
        "ASSINATURA": _texto(assinatura, 4000),
        "DETALHE": _texto(detalhe, 8000),
    }
    RESULTADOS.append(item)
    total_txt = "" if total is None else f" | {total:.3f}s"
    eq_txt = "" if equivalente is None else f" | eq={equivalente}"
    print(f"[LAB][{status}] {grupo} :: {teste}{total_txt}{eq_txt}")
    if detalhe:
        print("   ", _texto(detalhe, 900))
    return item

def safe_run(grupo, teste, fn, estrategia="", repeticao=None, prereq=True):
    inicio = time.perf_counter()
    if not prereq:
        registrar(grupo, teste, "SKIP", estrategia, repeticao, inicio, detalhe="pré-condição indisponível")
        return None
    try:
        out = fn()
        registrar(grupo, teste, "OK", estrategia, repeticao, inicio, assinatura=out)
        return out
    except Exception as exc:
        registrar(
            grupo, teste, "ERRO", estrategia, repeticao, inicio,
            detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}",
        )
        return None

def normalizar_valor(v):
    if isinstance(v, Decimal):
        return str(v)
    if isinstance(v, (datetime.date, datetime.datetime)):
        return v.isoformat()
    return v

def normalizar_rows(rows):
    saida = []
    for r in rows:
        d = r.asDict(recursive=True) if hasattr(r, "asDict") else dict(r)
        saida.append({k: normalizar_valor(v) for k, v in sorted(d.items())})
    return saida

def hash_rows(rows):
    payload = json.dumps(normalizar_rows(rows), ensure_ascii=False, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def plano(df, limite=16000):
    try:
        return df._jdf.queryExecution().executedPlan().toString()[:limite]
    except Exception as exc:
        return f"PLANO_INDISPONIVEL: {type(exc).__name__}: {exc}"

def n_particoes(df):
    try:
        return int(df.rdd.getNumPartitions())
    except Exception:
        return None

def recuar_um_mes(data):
    total = data.year * 12 + data.month - 2
    ano, mes0 = divmod(total, 12)
    mes = mes0 + 1
    return datetime.date(ano, mes, min(data.day, calendar.monthrange(ano, mes)[1]))

hoje_raw = os.environ.get("HOJE") or os.environ.get("CTMODATE") or os.environ.get("ctmodate")
try:
    DATA_EXECUCAO = datetime.date.fromisoformat(str(hoje_raw)[:10])
except Exception:
    DATA_EXECUCAO = datetime.date(2026, 9, 1)
    registrar("AMBIENTE", "DATA_EXECUCAO_FALLBACK", "WARN", inicio=time.perf_counter(),
              detalhe=f"Valor não interpretável: {hoje_raw!r}; fallback diagnóstico 2026-09-01")

DATA_INICIAL_PUBLICO = recuar_um_mes(DATA_EXECUCAO)
DATA_FINAL_EXCLUSIVA_PUBLICO = DATA_EXECUCAO

CONTEXTO.update({
    "DATA_EXECUCAO": DATA_EXECUCAO,
    "DATA_INICIAL_PUBLICO": DATA_INICIAL_PUBLICO,
    "DATA_FINAL_EXCLUSIVA_PUBLICO": DATA_FINAL_EXCLUSIVA_PUBLICO,
})

print("[LAB] CD_CLI:", CD_CLI)
print("[LAB] DATA_EXECUCAO:", DATA_EXECUCAO)
print("[LAB] JANELA_PUBLICO:", DATA_INICIAL_PUBLICO, "<= TS_INCL_TRAN <", DATA_FINAL_EXCLUSIVA_PUBLICO)

## 3.1 Gate de risco para fontes gigantes

**Execute primeiro com `MODO_LAB = "SEGURO"`**.

Esse primeiro passe já mede:

- baseline Q1/Q2/Q3/Q4/Q5;
- projeções seletivas;
- pushdown;
- fetchsize;
- lookup exato Q4 `(DT_REF, CD_CLI)`;
- busca regressiva Q4 por competência;
- `IN` de competências;
- microbenchmarks Spark;
- cache/lineage;
- resultado 1×80;
- transporte BBMagic.

Ele deliberadamente **não** dispara no primeiro passe:

- função/cast em `CD_CLI` na TRAN de ~27,9 bi;
- aritmética em `CD_CLI`;
- CTE/ROW_NUMBER/NOT EXISTS históricos na DVS de ~1,10 bi;
- `DISTINCT` histórico desnecessário;
- `OR` Q1+Q5;
- `SELECT *` experimental nas fontes gigantes;
- JDBC paralelo contra DB2.

Depois do primeiro relatório, mude somente:

```python
MODO_LAB = "AGRESSIVO"
```

para habilitar `ALTO`.

`EXTREMO` deve ser usado apenas se os resultados anteriores justificarem.

In [ ]:
%%spark

print("=" * 100)
print("GATE DE RISCO")
print("=" * 100)
print("MODO_LAB =", MODO_LAB)
print("Executa BAIXO:", permitido("BAIXO"))
print("Executa MEDIO:", permitido("MEDIO"))
print("Executa ALTO :", permitido("ALTO"))
print("Executa EXTREMO:", permitido("EXTREMO"))
print()
print("Fontes críticas:")
print("  DB2GFP.TRAN_RLZD_INST_PCT  ~27,9 bilhões de linhas")
print("  DB2D1D.DVS_GRDR_FNCO_PF    ~1,10 bilhão de linhas")
print()
print("Nenhum teste ALTO/EXTREMO será executado neste passe em modo SEGURO.")

## 4. Ambiente Spark efetivo — sem alterar configuração

In [ ]:
%%spark

inicio = time.perf_counter()
try:
    sc_conf = spark.sparkContext.getConf()
    chaves = [
        "spark.app.name",
        "spark.master",
        "spark.driver.memory",
        "spark.driver.cores",
        "spark.executor.instances",
        "spark.executor.memory",
        "spark.executor.cores",
        "spark.dynamicAllocation.enabled",
        "spark.dynamicAllocation.minExecutors",
        "spark.dynamicAllocation.maxExecutors",
        "spark.sql.adaptive.enabled",
        "spark.sql.adaptive.coalescePartitions.enabled",
        "spark.sql.shuffle.partitions",
        "spark.sql.autoBroadcastJoinThreshold",
        "spark.sql.session.timeZone",
        "spark.default.parallelism",
    ]
    confs = {}
    for k in chaves:
        try:
            confs[k] = spark.conf.get(k)
        except Exception:
            confs[k] = sc_conf.get(k, None)

    try:
        mem_status = spark.sparkContext._jsc.sc().getExecutorMemoryStatus().toString()
    except Exception as exc:
        mem_status = f"indisponível: {exc}"

    dados = {
        "spark_version": spark.version,
        "python": sys.version,
        "java": spark.sparkContext._jvm.java.lang.System.getProperty("java.version"),
        "master": spark.sparkContext.master,
        "application_id": spark.sparkContext.applicationId,
        "default_parallelism": spark.sparkContext.defaultParallelism,
        "platform": platform.platform(),
        "ambiente": os.environ.get("AMBIENTE"),
        "hoje": os.environ.get("HOJE"),
        "confs": confs,
        "executor_memory_status": mem_status,
    }
    CONTEXTO["AMBIENTE"] = dados
    registrar("AMBIENTE", "CONFIG_EFETIVA", "OK", inicio=inicio, assinatura=dados)
except Exception as exc:
    registrar("AMBIENTE", "CONFIG_EFETIVA", "ERRO", inicio=inicio,
              detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}")

## 5. Conector DB2 corporativo e assinatura real

In [ ]:
%%spark

inicio = time.perf_counter()
try:
    conector_db2 = criar_conector_db2_spark(env=dict(os.environ))
    CONTEXTO["CONECTOR_DB2_OK"] = True
    info = {
        "classe": type(conector_db2).__name__,
        "driver": getattr(conector_db2, "driver", None),
        "database": getattr(conector_db2, "database", None),
        "isolamento_padrao": getattr(conector_db2, "isolamento_padrao", None),
        "tem_sql": callable(getattr(conector_db2, "sql", None)),
        "tem_table": callable(getattr(conector_db2, "table", None)),
        "tem_list_columns": callable(getattr(conector_db2, "list_columns", None)),
    }
    registrar("CONECTOR", "CONECTOR_DB2", "OK", inicio=inicio, assinatura=info)
except Exception as exc:
    CONTEXTO["CONECTOR_DB2_OK"] = False
    registrar("CONECTOR", "CONECTOR_DB2", "ERRO", inicio=inicio,
              detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}")

## 6. Funções de benchmark DB2/Hive

In [ ]:
%%spark

def executar_db2(
    grupo,
    nome,
    sql,
    *,
    estrategia="",
    repeticoes=1,
    fetchsize=FETCHSIZE_BASE,
    timeout=TIMEOUT_NORMAL,
    isolation_level=None,
    partition_column=None,
    lower_bound=None,
    upper_bound=None,
    num_partitions=None,
    action="take",
    limit=20,
    baseline_hash=None,
):
    saidas = []
    for rep in range(1, repeticoes + 1):
        inicio = time.perf_counter()
        try:
            t0 = time.perf_counter()
            df = conector_db2.sql(
                sql,
                fetchsize=fetchsize,
                query_timeout=timeout,
                isolation_level=isolation_level,
                partition_column=partition_column,
                lower_bound=lower_bound,
                upper_bound=upper_bound,
                num_partitions=num_partitions,
            )
            prep = time.perf_counter() - t0
            parts = n_particoes(df)
            plan = plano(df, 6000)

            t1 = time.perf_counter()
            rows = []
            linhas = None
            assinatura_extra = None

            if action == "take":
                rows = df.take(limit)
                linhas = len(rows)
            elif action == "first":
                r = df.first()
                rows = [] if r is None else [r]
                linhas = len(rows)
            elif action == "count":
                linhas = int(df.count())
            elif action == "collect":
                rows = df.collect()
                linhas = len(rows)
            elif action == "agg_count_first":
                linhas = int(df.count())
                r = df.first()
                rows = [] if r is None else [r]
            else:
                raise ValueError(f"Action não suportada: {action}")

            action_s = time.perf_counter() - t1
            h = hash_rows(rows) if rows else None
            equivalente = None if baseline_hash is None or h is None else (h == baseline_hash)
            assinatura = {
                "hash": h,
                "rows": normalizar_rows(rows),
                "plano": plan,
                "assinatura_extra": assinatura_extra,
            }

            registrar(
                grupo, nome, "OK", estrategia, rep, inicio,
                preparacao=prep, action=action_s, linhas=linhas,
                colunas=len(df.columns), particoes=parts,
                fetchsize=fetchsize, isolamento=isolation_level,
                equivalente=equivalente, assinatura=assinatura,
            )
            saidas.append({"ok": True, "df": df, "rows": rows, "hash": h, "linhas": linhas})
        except Exception as exc:
            registrar(
                grupo, nome, "ERRO", estrategia, rep, inicio,
                fetchsize=fetchsize, isolamento=isolation_level,
                detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}",
            )
            saidas.append({"ok": False, "erro": str(exc)})
    return saidas

def executar_hive(
    grupo,
    nome,
    sql,
    *,
    estrategia="",
    repeticoes=1,
    action="take",
    limit=20,
    baseline_hash=None,
):
    saidas = []
    for rep in range(1, repeticoes + 1):
        inicio = time.perf_counter()
        try:
            t0 = time.perf_counter()
            df = spark.sql(sql)
            prep = time.perf_counter() - t0
            parts = n_particoes(df)
            plan = plano(df, 9000)

            t1 = time.perf_counter()
            rows = []
            linhas = None
            if action == "take":
                rows = df.take(limit)
                linhas = len(rows)
            elif action == "first":
                r = df.first()
                rows = [] if r is None else [r]
                linhas = len(rows)
            elif action == "count":
                linhas = int(df.count())
            elif action == "collect":
                rows = df.collect()
                linhas = len(rows)
            else:
                raise ValueError(action)
            action_s = time.perf_counter() - t1

            h = hash_rows(rows) if rows else None
            equivalente = None if baseline_hash is None or h is None else (h == baseline_hash)
            registrar(
                grupo, nome, "OK", estrategia, rep, inicio,
                preparacao=prep, action=action_s, linhas=linhas,
                colunas=len(df.columns), particoes=parts,
                equivalente=equivalente,
                assinatura={"hash": h, "rows": normalizar_rows(rows), "plano": plan},
            )
            saidas.append({"ok": True, "df": df, "rows": rows, "hash": h, "linhas": linhas})
        except Exception as exc:
            registrar(
                grupo, nome, "ERRO", estrategia, rep, inicio,
                detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}",
            )
            saidas.append({"ok": False, "erro": str(exc)})
    return saidas

## 7. Catálogo real: tabelas, cardinalidade estimada, colunas e índices

In [ ]:
%%spark

FONTES_DB2_META = [
    ("TRAN", "DB2GFP", "TRAN_RLZD_INST_PCT"),
    ("CICLO", "DB2GFP", "CT_GRDR_FNCO"),
    ("PERFIL", "DB2D1D", "DVS_GRDR_FNCO_PF"),
]

if not CONTEXTO.get("CONECTOR_DB2_OK"):
    registrar("CATALOGO", "DB2_META", "SKIP", inicio=time.perf_counter(), detalhe="Conector indisponível")
else:
    for rotulo, schema, tabela in FONTES_DB2_META:
        sql_tab = f'''
        SELECT CREATOR, NAME, TYPE, COLCOUNT, CARDF, NPAGES, FPAGES, STATS_TIME
        FROM SYSIBM.SYSTABLES
        WHERE CREATOR = '{schema}' AND NAME = '{tabela}'
        FETCH FIRST 5 ROWS ONLY
        '''
        executar_db2("CATALOGO", f"{rotulo}_SYSTABLES", sql_tab, timeout=60, action="take", limit=5)

        sql_cols = f'''
        SELECT NAME AS COLUNA, COLNO, COLTYPE, LENGTH, SCALE, NULLS
        FROM SYSIBM.SYSCOLUMNS
        WHERE TBCREATOR = '{schema}' AND TBNAME = '{tabela}'
        ORDER BY COLNO
        '''
        executar_db2("CATALOGO", f"{rotulo}_COLUNAS", sql_cols, timeout=60, action="take", limit=300)

        sql_idx = f'''
        SELECT I.TBCREATOR, I.TBNAME, I.NAME AS IXNAME, I.UNIQUERULE, K.COLNAME, K.COLSEQ
        FROM SYSIBM.SYSINDEXES I
        INNER JOIN SYSIBM.SYSKEYS K
          ON I.CREATOR = K.IXCREATOR
         AND I.NAME = K.IXNAME
        WHERE I.TBCREATOR = '{schema}'
          AND I.TBNAME = '{tabela}'
        ORDER BY I.NAME, K.COLSEQ
        '''
        executar_db2("CATALOGO", f"{rotulo}_INDICES", sql_idx, timeout=60, action="take", limit=300)

## 8. Metadados Hive da Q3

In [ ]:
%%spark

inicio = time.perf_counter()
try:
    fonte = FONTES["RENDA"]
    existe = spark.catalog.tableExists(fonte)
    if not existe:
        registrar("Q3_META", "TABELA_HIVE", "WARN", inicio=inicio, detalhe=f"{fonte} não encontrada")
    else:
        df = spark.table(fonte)
        schema = [(x.name, x.dataType.simpleString(), x.nullable) for x in df.schema.fields]
        try:
            particoes = [c.name for c in spark.catalog.listColumns(fonte) if getattr(c, "isPartition", False)]
        except Exception:
            particoes = []
        CONTEXTO["Q3_SCHEMA"] = schema
        CONTEXTO["Q3_PARTICOES"] = particoes
        registrar("Q3_META", "TABELA_HIVE", "OK", inicio=inicio,
                  assinatura={"schema": schema, "partition_cols": particoes})
except Exception as exc:
    registrar("Q3_META", "TABELA_HIVE", "ERRO", inicio=inicio,
              detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}")

## 9. Q1 baseline — formação do cliente e derivação de CPF/conta

In [ ]:
%%spark

Q1_SQL_BASE = f'''
SELECT
    CD_CLI,
    TS_INCL_TRAN,
    NR_CPF_CNPJ_TITR,
    NR_AG_TITR,
    CD_CT_TITR,
    NR_MCA_PCT_OPB,
    CD_PRD
FROM {FONTES["TRAN"]}
WHERE CD_CLI = {CD_CLI}
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
'''

out = executar_db2(
    "Q1", "Q1_BASELINE",
    Q1_SQL_BASE,
    repeticoes=REPETICOES_QUERY,
    timeout=TIMEOUT_NORMAL,
    action="count",
)

try:
    df_q1 = conector_db2.sql(Q1_SQL_BASE, fetchsize=FETCHSIZE_BASE, query_timeout=TIMEOUT_NORMAL)
    df_q1 = df_q1.persist(StorageLevel.MEMORY_AND_DISK)
    PERSISTIDOS.append(df_q1)
    qt_q1 = int(df_q1.count())

    resumo = df_q1.agg(
        F.max("TS_INCL_TRAN").alias("TS_REF"),
        F.countDistinct("NR_CPF_CNPJ_TITR").alias("QT_CPF"),
        F.max("NR_CPF_CNPJ_TITR").alias("CPF"),
    ).first()
    CONTEXTO["TS_INCL_TRAN_REF"] = resumo["TS_REF"]
    CONTEXTO["CD_CPF"] = resumo["CPF"] if int(resumo["QT_CPF"] or 0) == 1 else None

    contas = (
        df_q1
        .filter(
            (F.col("NR_MCA_PCT_OPB") == 999999999) &
            (F.col("CD_PRD") == 6) &
            F.col("NR_AG_TITR").isNotNull() &
            F.col("CD_CT_TITR").isNotNull()
        )
        .select("NR_AG_TITR", "CD_CT_TITR")
        .dropDuplicates()
        .limit(3)
        .collect()
    )

    conta = None
    if len(contas) == 1:
        ag = str(contas[0]["NR_AG_TITR"]).strip()
        cc = str(contas[0]["CD_CT_TITR"]).strip()
        if re.fullmatch(r"\d+", ag) and re.fullmatch(r"\d+", cc):
            conta = [int(ag), int(cc.lstrip("0") or "0")]

    CONTEXTO["CONTA"] = conta
    registrar("Q1", "Q1_DERIVACOES", "OK", inicio=time.perf_counter(),
              linhas=qt_q1, assinatura={"resumo": resumo.asDict(), "contas": normalizar_rows(contas), "conta": conta})
except Exception as exc:
    registrar("Q1", "Q1_DERIVACOES", "ERRO", inicio=time.perf_counter(),
              detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}")

## 10. Q1 — matriz de formulações SQL

In [ ]:
%%spark

Q1_VARIANTES = [
    ("Q1_A_BASE_TIMESTAMP", Q1_SQL_BASE, "baseline"),
    ("Q1_B_COMPARACAO_DIRETA", f'''
        SELECT CD_CLI, TS_INCL_TRAN, NR_CPF_CNPJ_TITR, NR_AG_TITR, CD_CT_TITR, NR_MCA_PCT_OPB, CD_PRD
        FROM {FONTES["TRAN"]}
        WHERE CD_CLI = {CD_CLI}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND TS_INCL_TRAN >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
          AND TS_INCL_TRAN < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
    ''', "sem função TIMESTAMP sobre a coluna"),
    ("Q1_C_DATE_NA_COLUNA", f'''
        SELECT CD_CLI, TS_INCL_TRAN, NR_CPF_CNPJ_TITR, NR_AG_TITR, CD_CT_TITR, NR_MCA_PCT_OPB, CD_PRD
        FROM {FONTES["TRAN"]}
        WHERE CD_CLI = {CD_CLI}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND DATE(TS_INCL_TRAN) >= DATE('{DATA_INICIAL_PUBLICO.isoformat()}')
          AND DATE(TS_INCL_TRAN) < DATE('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()}')
    ''', "função na coluna temporal — propositalmente suspeita"),
    ("Q1_D_CAST_CDCLI", f'''
        SELECT CD_CLI, TS_INCL_TRAN, NR_CPF_CNPJ_TITR, NR_AG_TITR, CD_CT_TITR, NR_MCA_PCT_OPB, CD_PRD
        FROM {FONTES["TRAN"]}
        WHERE BIGINT(CD_CLI) = BIGINT({CD_CLI})
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND TS_INCL_TRAN >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
          AND TS_INCL_TRAN < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
    ''', "cast/função na chave — propositalmente suspeita"),
    ("Q1_E_CDCLI_MAIS_ZERO", f'''
        SELECT CD_CLI, TS_INCL_TRAN, NR_CPF_CNPJ_TITR, NR_AG_TITR, CD_CT_TITR, NR_MCA_PCT_OPB, CD_PRD
        FROM {FONTES["TRAN"]}
        WHERE CD_CLI + 0 = {CD_CLI}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND TS_INCL_TRAN >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
          AND TS_INCL_TRAN < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
    ''', "aritmética na chave"),
    ("Q1_F_PREDICADOS_REORDENADOS", f'''
        SELECT CD_CLI, TS_INCL_TRAN, NR_CPF_CNPJ_TITR, NR_AG_TITR, CD_CT_TITR, NR_MCA_PCT_OPB, CD_PRD
        FROM {FONTES["TRAN"]}
        WHERE TS_INCL_TRAN < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
          AND CD_TIP_PSS = 1
          AND CD_EST_TRAN_INST = 0
          AND TS_INCL_TRAN >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
          AND CD_CLI = {CD_CLI}
    ''', "mesma semântica, ordem textual diferente"),
    ("Q1_G_PROJECAO_MINIMA", f'''
        SELECT CD_CLI, TS_INCL_TRAN, NR_CPF_CNPJ_TITR
        FROM {FONTES["TRAN"]}
        WHERE CD_CLI = {CD_CLI}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND TS_INCL_TRAN >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
          AND TS_INCL_TRAN < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
    ''', "projeção reduzida"),
    ("Q1_H_SELECT_STAR", f'''
        SELECT *
        FROM {FONTES["TRAN"]}
        WHERE CD_CLI = {CD_CLI}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND TS_INCL_TRAN >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
          AND TS_INCL_TRAN < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
    ''', "anti-padrão controlado: SELECT *"),
    ("Q1_I_AGREGACAO_DB2", f'''
        SELECT
            COUNT(*) AS QT,
            MAX(TS_INCL_TRAN) AS TS_REF,
            COUNT(DISTINCT NR_CPF_CNPJ_TITR) AS QT_CPF
        FROM {FONTES["TRAN"]}
        WHERE CD_CLI = {CD_CLI}
          AND CD_EST_TRAN_INST = 0
          AND CD_TIP_PSS = 1
          AND TS_INCL_TRAN >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
          AND TS_INCL_TRAN < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
    ''', "pushdown de agregação"),
]

RISCO_Q1 = {
    "Q1_A_BASE_TIMESTAMP": "BAIXO",
    "Q1_B_COMPARACAO_DIRETA": "BAIXO",
    "Q1_C_DATE_NA_COLUNA": "ALTO",
    "Q1_D_CAST_CDCLI": "EXTREMO",
    "Q1_E_CDCLI_MAIS_ZERO": "EXTREMO",
    "Q1_F_PREDICADOS_REORDENADOS": "BAIXO",
    "Q1_G_PROJECAO_MINIMA": "BAIXO",
    "Q1_H_SELECT_STAR": "ALTO",
    "Q1_I_AGREGACAO_DB2": "MEDIO",
}

for nome, sql, estrategia in Q1_VARIANTES:
    risco = RISCO_Q1[nome]
    if not permitido(risco):
        registrar_skip_risco("Q1_VARIANTES", nome, estrategia, risco)
        continue
    executar_db2(
        "Q1_VARIANTES", nome, sql,
        estrategia=f"[RISCO {risco}] {estrategia}",
        repeticoes=REPETICOES_QUERY,
        timeout=60 if risco == "MEDIO" else TIMEOUT_AGRESSIVO,
        action="take", limit=200
    )

## 11. Q1 — fetchsize e isolamento

In [ ]:
%%spark

for fs in [100, 500, 1000, 2500, 5000, 10000, 25000, 50000]:
    executar_db2(
        "Q1_FETCHSIZE", f"Q1_FETCH_{fs}", Q1_SQL_BASE,
        estrategia=f"fetchsize={fs}", repeticoes=1,
        fetchsize=fs, timeout=TIMEOUT_NORMAL, action="count",
    )

if EXECUTAR_TESTES_ISOLAMENTO:
    for iso in ["READ_UNCOMMITTED", "READ_COMMITTED"]:
        executar_db2(
            "Q1_ISOLAMENTO", f"Q1_ISO_{iso}", Q1_SQL_BASE,
            estrategia=iso, repeticoes=1, fetchsize=FETCHSIZE_BASE,
            timeout=90, isolation_level=iso, action="count",
        )

## 12. Q2 baseline e cálculo da janela

In [ ]:
%%spark

conta = CONTEXTO.get("CONTA")
if not conta:
    registrar("Q2", "Q2_BASELINE", "SKIP", inicio=time.perf_counter(), detalhe="Conta única indisponível")
else:
    ag, cc = conta
    Q2_SQL_BASE = f'''
    SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
    FROM {FONTES["CICLO"]}
    WHERE CD_UOR_CC = {ag}
      AND NR_CC = {cc}
    '''
    executar_db2("Q2", "Q2_BASELINE", Q2_SQL_BASE, repeticoes=REPETICOES_QUERY,
                 timeout=TIMEOUT_NORMAL, action="take", limit=200)

    try:
        df_q2 = conector_db2.sql(Q2_SQL_BASE, fetchsize=FETCHSIZE_BASE, query_timeout=TIMEOUT_NORMAL)
        rows = df_q2.orderBy(F.col("TS_ULT_EXEA_PSQ").desc()).limit(2).collect()
        linha = rows[0] if rows else None
        if linha:
            dia = int(linha["DD_INC_MM_CLC_BLC"]) if linha["DD_INC_MM_CLC_BLC"] is not None else 1
            ts_ref = CONTEXTO.get("TS_INCL_TRAN_REF")
            if ts_ref:
                dia_ref = min(dia, calendar.monthrange(ts_ref.year, ts_ref.month)[1])
                cand = datetime.datetime(ts_ref.year, ts_ref.month, dia_ref)
                if ts_ref >= cand:
                    ini_aberto = cand.date()
                else:
                    ini_aberto = recuar_um_mes(cand.date())
                fim = ini_aberto - timedelta(days=1)
                ini = recuar_um_mes(ini_aberto)
                CONTEXTO["DT_REF_INI"] = ini
                CONTEXTO["DT_REF_FIM"] = fim
                CONTEXTO["DIA_CICLO"] = dia
                registrar("Q2", "Q2_JANELA", "OK", inicio=time.perf_counter(),
                          assinatura={"linha": linha.asDict(), "DT_REF_INI": ini, "DT_REF_FIM": fim})
    except Exception as exc:
        registrar("Q2", "Q2_JANELA", "ERRO", inicio=time.perf_counter(),
                  detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}")

## 13. Q2 — matriz de seleção da linha mais recente

In [ ]:
%%spark

if CONTEXTO.get("CONTA"):
    ag, cc = CONTEXTO["CONTA"]
    Q2_VARIANTES = [
        ("Q2_A_TODAS_LINHAS", Q2_SQL_BASE, "trazer histórico filtrado e selecionar no Spark"),
        ("Q2_B_ORDER_FETCH1", f'''
            SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
            FROM {FONTES["CICLO"]}
            WHERE CD_UOR_CC = {ag} AND NR_CC = {cc}
            ORDER BY TS_ULT_EXEA_PSQ DESC
            FETCH FIRST 1 ROW ONLY
        ''', "pushdown ORDER BY + FETCH"),
        ("Q2_C_MAX_SUBQUERY", f'''
            SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
            FROM {FONTES["CICLO"]}
            WHERE CD_UOR_CC = {ag}
              AND NR_CC = {cc}
              AND TS_ULT_EXEA_PSQ = (
                  SELECT MAX(TS_ULT_EXEA_PSQ)
                  FROM {FONTES["CICLO"]}
                  WHERE CD_UOR_CC = {ag} AND NR_CC = {cc}
              )
            FETCH FIRST 1 ROW ONLY
        ''', "MAX + lookup na mesma query"),
        ("Q2_D_ROW_NUMBER", f'''
            SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
            FROM (
                SELECT C.*,
                       ROW_NUMBER() OVER (
                           PARTITION BY CD_UOR_CC, NR_CC
                           ORDER BY TS_ULT_EXEA_PSQ DESC
                       ) AS RN
                FROM {FONTES["CICLO"]} C
                WHERE CD_UOR_CC = {ag} AND NR_CC = {cc}
            ) X
            WHERE RN = 1
        ''', "ROW_NUMBER em subquery"),
        ("Q2_E_CTE_ROW_NUMBER", f'''
            WITH X AS (
                SELECT C.*,
                       ROW_NUMBER() OVER (
                           PARTITION BY CD_UOR_CC, NR_CC
                           ORDER BY TS_ULT_EXEA_PSQ DESC
                       ) AS RN
                FROM {FONTES["CICLO"]} C
                WHERE CD_UOR_CC = {ag} AND NR_CC = {cc}
            )
            SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
            FROM X
            WHERE RN = 1
        ''', "CTE — pode falhar no wrapper JDBC"),
        ("Q2_F_CAST_CC", f'''
            SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
            FROM {FONTES["CICLO"]}
            WHERE CD_UOR_CC = {ag}
              AND BIGINT(NR_CC) = BIGINT({cc})
            ORDER BY TS_ULT_EXEA_PSQ DESC
            FETCH FIRST 1 ROW ONLY
        ''', "função na chave"),
        ("Q2_G_PREDICADOS_REORDENADOS", f'''
            SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
            FROM {FONTES["CICLO"]}
            WHERE NR_CC = {cc} AND CD_UOR_CC = {ag}
            ORDER BY TS_ULT_EXEA_PSQ DESC
            FETCH FIRST 1 ROW ONLY
        ''', "ordem textual invertida"),
        ("Q2_H_MAX_INDEPENDENTE_ERRADO", f'''
            SELECT
                MAX(TS_ULT_EXEA_PSQ) AS TS_MAX,
                MAX(DD_INC_MM_CLC_BLC) AS DD_MAX
            FROM {FONTES["CICLO"]}
            WHERE CD_UOR_CC = {ag} AND NR_CC = {cc}
        ''', "controle negativo: pode violar mesma-linha"),
    ]

    for nome, sql, estrategia in Q2_VARIANTES:
        executar_db2("Q2_VARIANTES", nome, sql, estrategia=estrategia,
                     repeticoes=REPETICOES_QUERY if "CTE" not in nome else 1,
                     timeout=TIMEOUT_AGRESSIVO, action="take", limit=20)

## 14. Q3 baseline Hive

In [ ]:
%%spark

cpf = CONTEXTO.get("CD_CPF")
if cpf is None:
    registrar("Q3", "Q3_BASELINE", "SKIP", inicio=time.perf_counter(), detalhe="CPF único indisponível")
else:
    cpf_sql = str(int(Decimal(str(cpf))))
    Q3_SQL_BASE = f'''
    SELECT
        NR_CPF_BASE_SRF AS NR_CPF,
        DT_INCL_REN_AVLD,
        VL_REN
    FROM {FONTES["RENDA"]}
    WHERE NR_CPF_BASE_SRF = {cpf_sql}
    '''
    out_q3 = executar_hive("Q3", "Q3_BASELINE", Q3_SQL_BASE,
                           repeticoes=REPETICOES_QUERY, action="take", limit=500)

    try:
        df_q3 = spark.sql(Q3_SQL_BASE).persist(StorageLevel.MEMORY_AND_DISK)
        PERSISTIDOS.append(df_q3)
        qt = df_q3.count()
        row = df_q3.orderBy(F.col("DT_INCL_REN_AVLD").desc()).first()
        if row:
            CONTEXTO["RENDA_REF"] = row["DT_INCL_REN_AVLD"]
            CONTEXTO["RENDA"] = row["VL_REN"]
        registrar("Q3", "Q3_DERIVACAO", "OK", inicio=time.perf_counter(),
                  linhas=qt, assinatura=row.asDict() if row else None)
    except Exception as exc:
        registrar("Q3", "Q3_DERIVACAO", "ERRO", inicio=time.perf_counter(),
                  detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}")

## 15. Q3 — matriz Hive SQL

In [ ]:
%%spark

if cpf is not None:
    Q3_VARIANTES = [
        ("Q3_A_BASE_TODAS", Q3_SQL_BASE, "histórico filtrado"),
        ("Q3_B_ORDER_LIMIT1", f'''
            SELECT NR_CPF_BASE_SRF AS NR_CPF, DT_INCL_REN_AVLD, VL_REN
            FROM {FONTES["RENDA"]}
            WHERE NR_CPF_BASE_SRF = {cpf_sql}
            ORDER BY DT_INCL_REN_AVLD DESC
            LIMIT 1
        ''', "ORDER BY + LIMIT"),
        ("Q3_C_ROW_NUMBER", f'''
            SELECT NR_CPF, DT_INCL_REN_AVLD, VL_REN
            FROM (
              SELECT
                NR_CPF_BASE_SRF AS NR_CPF,
                DT_INCL_REN_AVLD,
                VL_REN,
                ROW_NUMBER() OVER (
                    PARTITION BY NR_CPF_BASE_SRF
                    ORDER BY DT_INCL_REN_AVLD DESC
                ) AS RN
              FROM {FONTES["RENDA"]}
              WHERE NR_CPF_BASE_SRF = {cpf_sql}
            ) X
            WHERE RN = 1
        ''', "ROW_NUMBER"),
        ("Q3_D_MAX_JOIN", f'''
            SELECT R.NR_CPF_BASE_SRF AS NR_CPF, R.DT_INCL_REN_AVLD, R.VL_REN
            FROM {FONTES["RENDA"]} R
            INNER JOIN (
                SELECT NR_CPF_BASE_SRF, MAX(DT_INCL_REN_AVLD) AS DT_MAX
                FROM {FONTES["RENDA"]}
                WHERE NR_CPF_BASE_SRF = {cpf_sql}
                GROUP BY NR_CPF_BASE_SRF
            ) M
              ON R.NR_CPF_BASE_SRF = M.NR_CPF_BASE_SRF
             AND R.DT_INCL_REN_AVLD = M.DT_MAX
            WHERE R.NR_CPF_BASE_SRF = {cpf_sql}
            LIMIT 1
        ''', "MAX + JOIN"),
        ("Q3_E_CTE_MAX", f'''
            WITH M AS (
                SELECT NR_CPF_BASE_SRF, MAX(DT_INCL_REN_AVLD) AS DT_MAX
                FROM {FONTES["RENDA"]}
                WHERE NR_CPF_BASE_SRF = {cpf_sql}
                GROUP BY NR_CPF_BASE_SRF
            )
            SELECT R.NR_CPF_BASE_SRF AS NR_CPF, R.DT_INCL_REN_AVLD, R.VL_REN
            FROM {FONTES["RENDA"]} R
            JOIN M
              ON R.NR_CPF_BASE_SRF = M.NR_CPF_BASE_SRF
             AND R.DT_INCL_REN_AVLD = M.DT_MAX
            LIMIT 1
        ''', "CTE Hive"),
        ("Q3_F_AGG_MAX_SOMENTE", f'''
            SELECT MAX(DT_INCL_REN_AVLD) AS DT_MAX
            FROM {FONTES["RENDA"]}
            WHERE NR_CPF_BASE_SRF = {cpf_sql}
        ''', "agregação redutora"),
        ("Q3_G_CAST_CPF", f'''
            SELECT NR_CPF_BASE_SRF AS NR_CPF, DT_INCL_REN_AVLD, VL_REN
            FROM {FONTES["RENDA"]}
            WHERE CAST(NR_CPF_BASE_SRF AS BIGINT) = CAST({cpf_sql} AS BIGINT)
            ORDER BY DT_INCL_REN_AVLD DESC
            LIMIT 1
        ''', "cast na chave"),
    ]

    RISCO_Q3 = {
        "Q3_A_BASE_TODAS": "BAIXO",
        "Q3_B_ORDER_LIMIT1": "BAIXO",
        "Q3_C_ROW_NUMBER": "MEDIO",
        "Q3_D_MAX_JOIN": "MEDIO",
        "Q3_E_CTE_MAX": "MEDIO",
        "Q3_F_AGG_MAX_SOMENTE": "BAIXO",
        "Q3_G_CAST_CPF": "ALTO",
    }

    for nome, sql, estrategia in Q3_VARIANTES:
        risco = RISCO_Q3[nome]
        if not permitido(risco):
            registrar_skip_risco("Q3_VARIANTES", nome, estrategia, risco)
            continue
        executar_hive(
            "Q3_VARIANTES", nome, sql,
            estrategia=f"[RISCO {risco}] {estrategia}",
            repeticoes=REPETICOES_QUERY, action="take", limit=50
        )

    # EXPLAIN FORMATTED das principais variantes.
    for nome, sql, estrategia in Q3_VARIANTES[:5]:
        executar_hive("Q3_EXPLAIN", f"EXPLAIN_{nome}",
                      "EXPLAIN FORMATTED " + sql,
                      estrategia=estrategia, repeticoes=1,
                      action="take", limit=200)

## 16. Q4 baseline funcional

In [ ]:
%%spark

Q4_SQL_BASE = f'''
SELECT
    CD_CLI,
    DT_REF,
    CD_MAC_PRFL_CLI,
    NM_MAC_PRFL_CLI,
    CD_MIC_PRFL_CLI,
    NM_MIC_PRFL_CLI
FROM {FONTES["PERFIL"]}
WHERE CD_CLI = {CD_CLI}
  AND DT_REF <= DATE('{DATA_EXECUCAO.isoformat()}')
ORDER BY DT_REF DESC
FETCH FIRST 1 ROW ONLY
'''

out_base_q4 = executar_db2(
    "Q4", "Q4_BASELINE_FUNCIONAL", Q4_SQL_BASE,
    estrategia="CD_CLI + DT_REF<=corte + ORDER BY DESC + FETCH 1",
    repeticoes=REPETICOES_QUERY,
    timeout=180, action="take", limit=2,
)

Q4_BASE_HASH = None
Q4_BASE_ROW = None
for o in out_base_q4:
    if o.get("ok") and o.get("rows"):
        Q4_BASE_HASH = o["hash"]
        Q4_BASE_ROW = normalizar_rows(o["rows"])[0]
        break

CONTEXTO["Q4_BASE_HASH"] = Q4_BASE_HASH
CONTEXTO["Q4_BASE_ROW"] = Q4_BASE_ROW
if Q4_BASE_ROW:
    CONTEXTO["Q4_DT_REF"] = Q4_BASE_ROW.get("DT_REF")
    print("[LAB] Q4 baseline:", Q4_BASE_ROW)

## 17. Q4 — matriz máxima de formulações SQL

In [ ]:
%%spark

dt_corte = DATA_EXECUCAO.isoformat()
dt_observada = str(CONTEXTO.get("Q4_DT_REF") or BASELINE_OBSERVADA["PERFIL_REF"])

Q4_VARIANTES = [
    ("Q4_A_BASE", Q4_SQL_BASE, "baseline"),
    ("Q4_B_SEM_CORTE_DATA", f'''
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]}
        WHERE CD_CLI = {CD_CLI}
        ORDER BY DT_REF DESC
        FETCH FIRST 1 ROW ONLY
    ''', "sem DT_REF<=DATA_EXECUCAO"),
    ("Q4_C_MAX_APENAS", f'''
        SELECT MAX(DT_REF) AS DT_REF
        FROM {FONTES["PERFIL"]}
        WHERE CD_CLI = {CD_CLI}
          AND DT_REF <= DATE('{dt_corte}')
    ''', "MAX por cliente"),
    ("Q4_D_MAX_SUBQUERY", f'''
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]}
        WHERE CD_CLI = {CD_CLI}
          AND DT_REF = (
              SELECT MAX(DT_REF)
              FROM {FONTES["PERFIL"]}
              WHERE CD_CLI = {CD_CLI}
                AND DT_REF <= DATE('{dt_corte}')
          )
        FETCH FIRST 1 ROW ONLY
    ''', "subquery escalar MAX"),
    ("Q4_E_JOIN_MAX", f'''
        SELECT P.CD_CLI, P.DT_REF, P.CD_MAC_PRFL_CLI, P.NM_MAC_PRFL_CLI, P.CD_MIC_PRFL_CLI, P.NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]} P
        INNER JOIN (
            SELECT CD_CLI, MAX(DT_REF) AS DT_MAX
            FROM {FONTES["PERFIL"]}
            WHERE CD_CLI = {CD_CLI}
              AND DT_REF <= DATE('{dt_corte}')
            GROUP BY CD_CLI
        ) M
          ON P.CD_CLI = M.CD_CLI
         AND P.DT_REF = M.DT_MAX
    ''', "MAX + JOIN"),
    ("Q4_F_ROW_NUMBER", f'''
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM (
            SELECT P.*,
                   ROW_NUMBER() OVER (PARTITION BY CD_CLI ORDER BY DT_REF DESC) AS RN
            FROM {FONTES["PERFIL"]} P
            WHERE CD_CLI = {CD_CLI}
              AND DT_REF <= DATE('{dt_corte}')
        ) X
        WHERE RN = 1
    ''', "ROW_NUMBER"),
    ("Q4_G_CTE_ROW_NUMBER", f'''
        WITH X AS (
            SELECT P.*,
                   ROW_NUMBER() OVER (PARTITION BY CD_CLI ORDER BY DT_REF DESC) AS RN
            FROM {FONTES["PERFIL"]} P
            WHERE CD_CLI = {CD_CLI}
              AND DT_REF <= DATE('{dt_corte}')
        )
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM X
        WHERE RN = 1
    ''', "CTE + ROW_NUMBER; wrapper pode rejeitar"),
    ("Q4_H_CTE_MAX", f'''
        WITH M AS (
            SELECT CD_CLI, MAX(DT_REF) AS DT_MAX
            FROM {FONTES["PERFIL"]}
            WHERE CD_CLI = {CD_CLI}
              AND DT_REF <= DATE('{dt_corte}')
            GROUP BY CD_CLI
        )
        SELECT P.CD_CLI, P.DT_REF, P.CD_MAC_PRFL_CLI, P.NM_MAC_PRFL_CLI, P.CD_MIC_PRFL_CLI, P.NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]} P
        JOIN M ON P.CD_CLI=M.CD_CLI AND P.DT_REF=M.DT_MAX
    ''', "CTE + MAX; wrapper pode rejeitar"),
    ("Q4_I_NOT_EXISTS", f'''
        SELECT P.CD_CLI, P.DT_REF, P.CD_MAC_PRFL_CLI, P.NM_MAC_PRFL_CLI, P.CD_MIC_PRFL_CLI, P.NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]} P
        WHERE P.CD_CLI = {CD_CLI}
          AND P.DT_REF <= DATE('{dt_corte}')
          AND NOT EXISTS (
              SELECT 1
              FROM {FONTES["PERFIL"]} P2
              WHERE P2.CD_CLI = P.CD_CLI
                AND P2.DT_REF <= DATE('{dt_corte}')
                AND P2.DT_REF > P.DT_REF
          )
        FETCH FIRST 1 ROW ONLY
    ''', "anti-join temporal NOT EXISTS"),
    ("Q4_J_EXATA_DTREF_CDCLI", f'''
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]}
        WHERE DT_REF = DATE('{dt_observada}')
          AND CD_CLI = {CD_CLI}
        FETCH FIRST 2 ROWS ONLY
    ''', "lookup alinhado à PK (DT_REF,CD_CLI)"),
    ("Q4_K_EXATA_PREDICADO_INVERSO", f'''
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]}
        WHERE CD_CLI = {CD_CLI}
          AND DT_REF = DATE('{dt_observada}')
        FETCH FIRST 2 ROWS ONLY
    ''', "mesma igualdade, ordem textual invertida"),
    ("Q4_L_BETWEEN_HISTORICO", f'''
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]}
        WHERE CD_CLI = {CD_CLI}
          AND DT_REF BETWEEN DATE('2000-01-01') AND DATE('{dt_corte}')
        ORDER BY DT_REF DESC
        FETCH FIRST 1 ROW ONLY
    ''', "BETWEEN histórico"),
    ("Q4_M_CHAR_DTREF", f'''
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]}
        WHERE CD_CLI = {CD_CLI}
          AND CHAR(DT_REF, ISO) <= '{dt_corte}'
        ORDER BY DT_REF DESC
        FETCH FIRST 1 ROW ONLY
    ''', "função CHAR na coluna líder"),
    ("Q4_N_BIGINT_CDCLI", f'''
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]}
        WHERE BIGINT(CD_CLI) = BIGINT({CD_CLI})
          AND DT_REF <= DATE('{dt_corte}')
        ORDER BY DT_REF DESC
        FETCH FIRST 1 ROW ONLY
    ''', "função/cast em CD_CLI"),
    ("Q4_O_CDCLI_MAIS_ZERO", f'''
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]}
        WHERE CD_CLI + 0 = {CD_CLI}
          AND DT_REF <= DATE('{dt_corte}')
        ORDER BY DT_REF DESC
        FETCH FIRST 1 ROW ONLY
    ''', "aritmética na chave"),
    ("Q4_P_DISTINCT", f'''
        SELECT DISTINCT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]}
        WHERE CD_CLI = {CD_CLI}
          AND DT_REF <= DATE('{dt_corte}')
        ORDER BY DT_REF DESC
        FETCH FIRST 1 ROW ONLY
    ''', "DISTINCT propositalmente desnecessário"),
    ("Q4_Q_GLOBAL_MAX_DTREF", f'''
        SELECT MAX(DT_REF) AS DT_REF
        FROM {FONTES["PERFIL"]}
        WHERE DT_REF <= DATE('{dt_corte}')
    ''', "MAX global pela coluna líder — não equivale ao último perfil do cliente"),
]

RISCO_Q4 = {
    "Q4_A_BASE": "MEDIO",                 # já observado ~94s no mesmo cliente
    "Q4_B_SEM_CORTE_DATA": "ALTO",
    "Q4_C_MAX_APENAS": "ALTO",
    "Q4_D_MAX_SUBQUERY": "ALTO",
    "Q4_E_JOIN_MAX": "ALTO",
    "Q4_F_ROW_NUMBER": "EXTREMO",
    "Q4_G_CTE_ROW_NUMBER": "EXTREMO",
    "Q4_H_CTE_MAX": "ALTO",
    "Q4_I_NOT_EXISTS": "EXTREMO",
    "Q4_J_EXATA_DTREF_CDCLI": "BAIXO",    # alinhada à PK (DT_REF, CD_CLI)
    "Q4_K_EXATA_PREDICADO_INVERSO": "BAIXO",
    "Q4_L_BETWEEN_HISTORICO": "ALTO",
    "Q4_M_CHAR_DTREF": "EXTREMO",
    "Q4_N_BIGINT_CDCLI": "EXTREMO",
    "Q4_O_CDCLI_MAIS_ZERO": "EXTREMO",
    "Q4_P_DISTINCT": "ALTO",
    "Q4_Q_GLOBAL_MAX_DTREF": "ALTO",
}

for nome, sql, estrategia in Q4_VARIANTES:
    risco = RISCO_Q4[nome]
    if not permitido(risco):
        registrar_skip_risco("Q4_VARIANTES", nome, estrategia, risco)
        continue
    executar_db2(
        "Q4_VARIANTES", nome, sql,
        estrategia=f"[RISCO {risco}] {estrategia}",
        repeticoes=1,
        timeout=120 if nome == "Q4_A_BASE" else 45,
        action="take",
        limit=5,
        baseline_hash=Q4_BASE_HASH if nome not in {"Q4_C_MAX_APENAS", "Q4_Q_GLOBAL_MAX_DTREF"} else None,
    )

## 18. Q4 — busca regressiva por competência exata

In [ ]:
%%spark

inicio_busca = time.perf_counter()
encontrado = None
tentativas = []

try:
    # Começa no primeiro dia do mês da DATA_EXECUCAO e recua mês a mês.
    ref = DATA_EXECUCAO.replace(day=1)
    for i in range(0, 60):
        sql = f'''
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]}
        WHERE DT_REF = DATE('{ref.isoformat()}')
          AND CD_CLI = {CD_CLI}
        FETCH FIRST 2 ROWS ONLY
        '''
        t = time.perf_counter()
        try:
            df = conector_db2.sql(sql, fetchsize=100, query_timeout=60)
            rows = df.take(2)
            dur = time.perf_counter() - t
            tentativas.append({"dt_ref": ref.isoformat(), "s": dur, "linhas": len(rows)})
            if rows:
                encontrado = rows[0]
                break
        except Exception as exc:
            tentativas.append({"dt_ref": ref.isoformat(), "s": time.perf_counter()-t, "erro": str(exc)})
        ref = recuar_um_mes(ref)

    h = hash_rows([encontrado]) if encontrado is not None else None
    registrar(
        "Q4_BUSCA_REGRESSIVA",
        "Q4_COMPETENCIAS_EXATAS",
        "OK" if encontrado is not None else "WARN",
        estrategia="lookup exato (DT_REF,CD_CLI) mês a mês",
        inicio=inicio_busca,
        linhas=1 if encontrado is not None else 0,
        equivalente=None if h is None or Q4_BASE_HASH is None else h == Q4_BASE_HASH,
        assinatura={
            "encontrado": normalizar_rows([encontrado]) if encontrado is not None else [],
            "tentativas": tentativas,
        },
    )
except Exception as exc:
    registrar("Q4_BUSCA_REGRESSIVA", "Q4_COMPETENCIAS_EXATAS", "ERRO", inicio=inicio_busca,
              detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}")

## 19. Q4 — teste com lista IN e UNION ALL de competências

In [ ]:
%%spark

refs = []
ref = DATA_EXECUCAO.replace(day=1)
for _ in range(18):
    refs.append(ref)
    ref = recuar_um_mes(ref)

lista_in = ", ".join(f"DATE('{d.isoformat()}')" for d in refs)

sql_in = f'''
SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
FROM {FONTES["PERFIL"]}
WHERE DT_REF IN ({lista_in})
  AND CD_CLI = {CD_CLI}
ORDER BY DT_REF DESC
FETCH FIRST 1 ROW ONLY
'''
if permitido("MEDIO"):
    executar_db2("Q4_VARIANTES", "Q4_R_IN_18_COMPETENCIAS", sql_in,
                 estrategia="[RISCO MEDIO] IN de competências exatas", timeout=45,
                 action="take", limit=2, baseline_hash=Q4_BASE_HASH)
else:
    registrar_skip_risco("Q4_VARIANTES", "Q4_R_IN_18_COMPETENCIAS", "IN de competências exatas", "MEDIO")

blocos = []
for d in refs:
    blocos.append(f'''
        SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
        FROM {FONTES["PERFIL"]}
        WHERE DT_REF = DATE('{d.isoformat()}') AND CD_CLI = {CD_CLI}
    ''')
sql_union = " UNION ALL ".join(blocos)
sql_union = f'''
SELECT *
FROM (
{sql_union}
) U
ORDER BY DT_REF DESC
FETCH FIRST 1 ROW ONLY
'''
if permitido("ALTO"):
    executar_db2("Q4_VARIANTES", "Q4_S_UNION_ALL_18_COMPETENCIAS", sql_union,
                 estrategia="[RISCO ALTO] UNION ALL de lookups exatos", timeout=60,
                 action="take", limit=2, baseline_hash=Q4_BASE_HASH)
else:
    registrar_skip_risco("Q4_VARIANTES", "Q4_S_UNION_ALL_18_COMPETENCIAS", "UNION ALL de lookups exatos", "ALTO")

## 20. Q4 — JDBC particionado usando o mesmo ConectorDb2Spark

In [ ]:
%%spark

if EXECUTAR_TESTES_JDBC_PARTICIONADO:
    # Testa somente consulta já restrita a um cliente. Bounds são de DT_REF.
    sql_q4_part = f'''
    SELECT
        CD_CLI, DT_REF,
        CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI,
        CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
    FROM {FONTES["PERFIL"]}
    WHERE CD_CLI = {CD_CLI}
      AND DT_REF >= DATE('2020-01-01')
      AND DT_REF <= DATE('{DATA_EXECUCAO.isoformat()}')
    '''
    for np in [2, 4]:
        executar_db2(
            "Q4_JDBC_PART",
            f"Q4_PART_DTREF_{np}",
            sql_q4_part,
            estrategia=f"partitionColumn=DT_REF; numPartitions={np}",
            repeticoes=1,
            timeout=120,
            partition_column="DT_REF",
            lower_bound="2020-01-01",
            upper_bound=(DATA_EXECUCAO + timedelta(days=1)).isoformat(),
            num_partitions=np,
            action="count",
        )

## 21. Q5 baseline

In [ ]:
%%spark

dt_ini = CONTEXTO.get("DT_REF_INI")
dt_fim = CONTEXTO.get("DT_REF_FIM")

if not (dt_ini and dt_fim):
    registrar("Q5", "Q5_BASELINE", "SKIP", inicio=time.perf_counter(), detalhe="Janela financeira indisponível")
else:
    dt_ctx_ini = dt_ini - timedelta(days=DIAS_CONTEXTO_RECONCILIACAO)
    dt_ctx_fim = dt_fim + timedelta(days=DIAS_CONTEXTO_RECONCILIACAO)

    Q5_SQL_BASE = f'''
    SELECT
        NR_TRAN_INST_PCT,
        CD_CLI,
        DT_TRAN,
        CD_NTZ_CTB_TRAN,
        CD_CTGR_TRAN_OGNL,
        CD_TIP_MOE_CRR,
        VL_TRAN
    FROM {FONTES["TRAN"]}
    WHERE CD_CLI = {CD_CLI}
      AND DT_TRAN >= DATE('{dt_ctx_ini.isoformat()}')
      AND DT_TRAN <= DATE('{dt_ctx_fim.isoformat()}')
      AND CD_EST_TRAN_INST = 0
    '''
    executar_db2("Q5", "Q5_BASELINE", Q5_SQL_BASE, repeticoes=REPETICOES_QUERY,
                 timeout=TIMEOUT_NORMAL, action="count")

    try:
        df_q5 = conector_db2.sql(Q5_SQL_BASE, fetchsize=FETCHSIZE_BASE, query_timeout=TIMEOUT_NORMAL)
        df_q5 = df_q5.persist(StorageLevel.MEMORY_AND_DISK)
        PERSISTIDOS.append(df_q5)
        qt_ctx = df_q5.count()
        df_q5_oficial = df_q5.filter((F.col("DT_TRAN") >= F.lit(dt_ini)) & (F.col("DT_TRAN") <= F.lit(dt_fim)))
        qt_oficial = df_q5_oficial.count()
        CONTEXTO["Q5_QT_CTX"] = qt_ctx
        CONTEXTO["Q5_QT_OFICIAL"] = qt_oficial
        CONTEXTO["DF_Q5"] = "df_q5"
        registrar("Q5", "Q5_RECORTE", "OK", inicio=time.perf_counter(),
                  assinatura={"contexto": qt_ctx, "oficial": qt_oficial,
                              "janela_contexto": [dt_ctx_ini, dt_ctx_fim],
                              "janela_oficial": [dt_ini, dt_fim]})
    except Exception as exc:
        registrar("Q5", "Q5_RECORTE", "ERRO", inicio=time.perf_counter(),
                  detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}")

## 22. Q5 — matriz SQL

In [ ]:
%%spark

if dt_ini and dt_fim:
    Q5_VARIANTES = [
        ("Q5_A_BASE", Q5_SQL_BASE, "baseline halo ±5"),
        ("Q5_B_BETWEEN", f'''
            SELECT NR_TRAN_INST_PCT, CD_CLI, DT_TRAN, CD_NTZ_CTB_TRAN, CD_CTGR_TRAN_OGNL, CD_TIP_MOE_CRR, VL_TRAN
            FROM {FONTES["TRAN"]}
            WHERE CD_CLI = {CD_CLI}
              AND DT_TRAN BETWEEN DATE('{dt_ctx_ini.isoformat()}') AND DATE('{dt_ctx_fim.isoformat()}')
              AND CD_EST_TRAN_INST = 0
        ''', "BETWEEN"),
        ("Q5_C_OFICIAL_PUSH_ORIGEM", f'''
            SELECT NR_TRAN_INST_PCT, CD_CLI, DT_TRAN, CD_NTZ_CTB_TRAN, CD_CTGR_TRAN_OGNL, CD_TIP_MOE_CRR, VL_TRAN
            FROM {FONTES["TRAN"]}
            WHERE CD_CLI = {CD_CLI}
              AND DT_TRAN >= DATE('{dt_ini.isoformat()}')
              AND DT_TRAN <= DATE('{dt_fim.isoformat()}')
              AND CD_EST_TRAN_INST = 0
        ''', "controle: só janela oficial, não substitui halo funcional"),
        ("Q5_D_PROJECAO_MINIMA", f'''
            SELECT NR_TRAN_INST_PCT, DT_TRAN, CD_NTZ_CTB_TRAN, VL_TRAN
            FROM {FONTES["TRAN"]}
            WHERE CD_CLI = {CD_CLI}
              AND DT_TRAN >= DATE('{dt_ctx_ini.isoformat()}')
              AND DT_TRAN <= DATE('{dt_ctx_fim.isoformat()}')
              AND CD_EST_TRAN_INST = 0
        ''', "projeção mínima"),
        ("Q5_E_SELECT_STAR", f'''
            SELECT *
            FROM {FONTES["TRAN"]}
            WHERE CD_CLI = {CD_CLI}
              AND DT_TRAN >= DATE('{dt_ctx_ini.isoformat()}')
              AND DT_TRAN <= DATE('{dt_ctx_fim.isoformat()}')
              AND CD_EST_TRAN_INST = 0
        ''', "SELECT * controlado"),
        ("Q5_F_CAST_CDCLI", f'''
            SELECT NR_TRAN_INST_PCT, CD_CLI, DT_TRAN, CD_NTZ_CTB_TRAN, CD_CTGR_TRAN_OGNL, CD_TIP_MOE_CRR, VL_TRAN
            FROM {FONTES["TRAN"]}
            WHERE BIGINT(CD_CLI) = BIGINT({CD_CLI})
              AND DT_TRAN >= DATE('{dt_ctx_ini.isoformat()}')
              AND DT_TRAN <= DATE('{dt_ctx_fim.isoformat()}')
              AND CD_EST_TRAN_INST = 0
        ''', "cast/função na chave"),
        ("Q5_G_PREDICADOS_REORDENADOS", f'''
            SELECT NR_TRAN_INST_PCT, CD_CLI, DT_TRAN, CD_NTZ_CTB_TRAN, CD_CTGR_TRAN_OGNL, CD_TIP_MOE_CRR, VL_TRAN
            FROM {FONTES["TRAN"]}
            WHERE DT_TRAN <= DATE('{dt_ctx_fim.isoformat()}')
              AND CD_EST_TRAN_INST = 0
              AND CD_CLI = {CD_CLI}
              AND DT_TRAN >= DATE('{dt_ctx_ini.isoformat()}')
        ''', "ordem textual"),
        ("Q5_H_AGG_DB2", f'''
            SELECT
                COUNT(*) AS QT,
                MIN(DT_TRAN) AS DT_MIN,
                MAX(DT_TRAN) AS DT_MAX,
                SUM(CASE WHEN CD_NTZ_CTB_TRAN='C' THEN 1 ELSE 0 END) AS QT_C,
                SUM(CASE WHEN CD_NTZ_CTB_TRAN='D' THEN 1 ELSE 0 END) AS QT_D,
                SUM(VL_TRAN) AS VL_TOTAL
            FROM {FONTES["TRAN"]}
            WHERE CD_CLI = {CD_CLI}
              AND DT_TRAN >= DATE('{dt_ctx_ini.isoformat()}')
              AND DT_TRAN <= DATE('{dt_ctx_fim.isoformat()}')
              AND CD_EST_TRAN_INST = 0
        ''', "agregação pushdown"),
    ]

    RISCO_Q5 = {
        "Q5_A_BASE": "BAIXO",
        "Q5_B_BETWEEN": "BAIXO",
        "Q5_C_OFICIAL_PUSH_ORIGEM": "BAIXO",
        "Q5_D_PROJECAO_MINIMA": "BAIXO",
        "Q5_E_SELECT_STAR": "ALTO",
        "Q5_F_CAST_CDCLI": "EXTREMO",
        "Q5_G_PREDICADOS_REORDENADOS": "BAIXO",
        "Q5_H_AGG_DB2": "MEDIO",
    }

    for nome, sql, estrategia in Q5_VARIANTES:
        risco = RISCO_Q5[nome]
        if not permitido(risco):
            registrar_skip_risco("Q5_VARIANTES", nome, estrategia, risco)
            continue
        executar_db2(
            "Q5_VARIANTES", nome, sql,
            estrategia=f"[RISCO {risco}] {estrategia}",
            repeticoes=REPETICOES_QUERY,
            timeout=60, action="take", limit=500
        )

## 23. Q5 — fetchsize

In [ ]:
%%spark

if dt_ini and dt_fim:
    for fs in [50, 100, 500, 1000, 2500, 5000, 10000, 25000, 50000]:
        executar_db2(
            "Q5_FETCHSIZE", f"Q5_FETCH_{fs}", Q5_SQL_BASE,
            estrategia=f"fetchsize={fs}", fetchsize=fs,
            timeout=90, repeticoes=1, action="count",
        )

## 24. Q5 — bounds e JDBC particionado por NR_TRAN_INST_PCT

In [ ]:
%%spark

if dt_ini and dt_fim and EXECUTAR_TESTES_JDBC_PARTICIONADO:
    sql_bounds = f'''
    SELECT MIN(NR_TRAN_INST_PCT) AS MIN_ID, MAX(NR_TRAN_INST_PCT) AS MAX_ID
    FROM {FONTES["TRAN"]}
    WHERE CD_CLI = {CD_CLI}
      AND DT_TRAN >= DATE('{dt_ctx_ini.isoformat()}')
      AND DT_TRAN <= DATE('{dt_ctx_fim.isoformat()}')
      AND CD_EST_TRAN_INST = 0
    '''
    bounds_out = executar_db2("Q5_JDBC_PART", "Q5_BOUNDS_ID", sql_bounds,
                              estrategia="bounds para partitionColumn", timeout=60,
                              action="take", limit=2)
    try:
        br = next(o for o in bounds_out if o.get("ok") and o.get("rows"))
        row = br["rows"][0]
        lb = int(row["MIN_ID"])
        ub = int(row["MAX_ID"]) + 1
        if lb < ub:
            for np in [2, 4, 8]:
                executar_db2(
                    "Q5_JDBC_PART", f"Q5_PART_ID_{np}", Q5_SQL_BASE,
                    estrategia=f"partitionColumn=NR_TRAN_INST_PCT; numPartitions={np}",
                    timeout=90, repeticoes=1,
                    partition_column="NR_TRAN_INST_PCT",
                    lower_bound=lb,
                    upper_bound=ub,
                    num_partitions=np,
                    action="count",
                )
    except Exception as exc:
        registrar("Q5_JDBC_PART", "Q5_PART_SETUP", "ERRO", inicio=time.perf_counter(),
                  detalhe=f"{type(exc).__name__}: {exc}")

## 25. Teste experimental: Q1 e Q5 em uma única leitura

In [ ]:
%%spark

if dt_ini and dt_fim:
    menor_ts = min(datetime.datetime.combine(DATA_INICIAL_PUBLICO, datetime.time.min),
                   datetime.datetime.combine(dt_ctx_ini, datetime.time.min))
    maior_ts = datetime.datetime.combine(max(DATA_FINAL_EXCLUSIVA_PUBLICO, dt_ctx_fim + timedelta(days=1)), datetime.time.min)

    sql_unificada = f'''
    SELECT
        NR_TRAN_INST_PCT,
        CD_CLI,
        TS_INCL_TRAN,
        NR_CPF_CNPJ_TITR,
        NR_AG_TITR,
        CD_CT_TITR,
        NR_MCA_PCT_OPB,
        CD_PRD,
        CD_EST_TRAN_INST,
        CD_TIP_PSS,
        DT_TRAN,
        CD_NTZ_CTB_TRAN,
        CD_CTGR_TRAN_OGNL,
        CD_TIP_MOE_CRR,
        VL_TRAN
    FROM {FONTES["TRAN"]}
    WHERE CD_CLI = {CD_CLI}
      AND CD_EST_TRAN_INST = 0
      AND (
          (
            CD_TIP_PSS = 1
            AND TS_INCL_TRAN >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
            AND TS_INCL_TRAN < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
          )
          OR
          (
            DT_TRAN >= DATE('{dt_ctx_ini.isoformat()}')
            AND DT_TRAN <= DATE('{dt_ctx_fim.isoformat()}')
          )
      )
    '''
    if permitido("ALTO"):
        executar_db2(
            "TRAN_UNIFICADA", "Q1_Q5_UNICA_LEITURA",
            sql_unificada,
            estrategia="[RISCO ALTO] OR das necessidades Q1 e Q5 em uma leitura",
            timeout=60, repeticoes=1, action="count",
        )
    else:
        registrar_skip_risco(
            "TRAN_UNIFICADA", "Q1_Q5_UNICA_LEITURA",
            "OR das necessidades Q1 e Q5 em uma leitura", "ALTO"
        )

## 26. Reconciliação no volume real — custo por operação

In [ ]:
%%spark

try:
    df_real = df_q5_oficial
    CHAVE = ["CD_CLI", "DT_TRAN", "VL_TRAN", "CD_TIP_MOE_CRR"]

    t = time.perf_counter()
    elegivel = (
        F.col("CD_NTZ_CTB_TRAN").isin("C", "D") &
        F.col("DT_TRAN").isNotNull() &
        F.col("VL_TRAN").isNotNull() &
        F.col("CD_TIP_MOE_CRR").isNotNull()
    )
    chaves = (
        df_real.filter(elegivel)
        .groupBy(*CHAVE)
        .agg(
            F.sum(F.when(F.col("CD_NTZ_CTB_TRAN") == "C", 1).otherwise(0)).alias("QT_C"),
            F.sum(F.when(F.col("CD_NTZ_CTB_TRAN") == "D", 1).otherwise(0)).alias("QT_D"),
        )
        .withColumn("QT_PARES", F.least("QT_C", "QT_D"))
        .filter(F.col("QT_PARES") > 0)
    )
    resumo = chaves.agg(F.count("*").alias("CHAVES"), F.sum("QT_PARES").alias("PARES")).first()
    registrar("RECONCILIACAO", "GROUPBY_PARES", "OK", inicio=t, assinatura=resumo.asDict())

    pares = int(resumo["PARES"] or 0)
    if pares:
        t = time.perf_counter()
        w = Window.partitionBy(*(CHAVE + ["CD_NTZ_CTB_TRAN"])).orderBy(F.col("NR_TRAN_INST_PCT"))
        marcado = (
            df_real.join(chaves.select(*CHAVE, "QT_PARES"), CHAVE, "left")
            .withColumn("_RN", F.row_number().over(w))
            .withColumn("_REMOVE",
                F.col("QT_PARES").isNotNull() &
                F.col("CD_NTZ_CTB_TRAN").isin("C","D") &
                (F.col("_RN") <= F.col("QT_PARES"))
            )
        )
        rem = marcado.filter("_REMOVE").count()
        registrar("RECONCILIACAO", "JOIN_WINDOW_REMOVE", "OK", inicio=t,
                  assinatura={"pares": pares, "linhas_removidas": rem})
except Exception as exc:
    registrar("RECONCILIACAO", "REAL", "ERRO", inicio=time.perf_counter(),
              detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}")

## 27. Scheduler Spark — custo mínimo de uma action

In [ ]:
%%spark

for i in range(1, 11):
    inicio = time.perf_counter()
    try:
        t = time.perf_counter()
        df = spark.range(1)
        prep = time.perf_counter() - t
        t = time.perf_counter()
        n = df.count()
        act = time.perf_counter() - t
        registrar("SPARK_SCHEDULER", "RANGE1_COUNT", "OK", repeticao=i, inicio=inicio,
                  preparacao=prep, action=act, linhas=n, colunas=1, particoes=n_particoes(df))
    except Exception as exc:
        registrar("SPARK_SCHEDULER", "RANGE1_COUNT", "ERRO", repeticao=i, inicio=inicio, detalhe=str(exc))

## 28. Cache e lineage — sintético e real

In [ ]:
%%spark

# Sintético com transformação suficiente para criar shuffle.
try:
    base = spark.range(0, 500000).select(
        "id",
        (F.col("id") % 101).alias("grp"),
        (F.col("id") * 1.0).alias("valor"),
    )
    deriv = base.filter((F.col("id") % 3) == 0).groupBy("grp").agg(
        F.sum("valor").alias("soma"), F.count("*").alias("n")
    )

    for i in range(1, 3):
        t = time.perf_counter()
        n = deriv.count()
        registrar("CACHE_LINEAGE", "SEM_CACHE", "OK", repeticao=i, inicio=t, linhas=n)

    for nivel_nome, nivel in [
        ("MEMORY_ONLY", StorageLevel.MEMORY_ONLY),
        ("MEMORY_AND_DISK", StorageLevel.MEMORY_AND_DISK),
        ("DISK_ONLY", StorageLevel.DISK_ONLY),
    ]:
        d = deriv.persist(nivel)
        PERSISTIDOS.append(d)
        t = time.perf_counter()
        n1 = d.count()
        registrar("CACHE_LINEAGE", f"{nivel_nome}_MATERIALIZA", "OK", inicio=t, linhas=n1)
        t = time.perf_counter()
        n2 = d.count()
        registrar("CACHE_LINEAGE", f"{nivel_nome}_HIT", "OK", inicio=t, linhas=n2)
        d.unpersist(blocking=False)
except Exception as exc:
    registrar("CACHE_LINEAGE", "SINTETICO", "ERRO", inicio=time.perf_counter(),
              detalhe=f"{type(exc).__name__}: {exc}")

# Reuso real da Q5 já pequena.
try:
    for nome_action, fn in [
        ("count", lambda: df_q5_oficial.count()),
        ("first", lambda: df_q5_oficial.first()),
        ("take20", lambda: df_q5_oficial.take(20)),
        ("agg", lambda: df_q5_oficial.agg(F.count("*").alias("N"), F.sum("VL_TRAN").alias("S")).first()),
        ("groupby", lambda: df_q5_oficial.groupBy("CD_NTZ_CTB_TRAN").count().collect()),
        ("order_limit", lambda: df_q5_oficial.orderBy("DT_TRAN").limit(20).collect()),
    ]:
        t = time.perf_counter()
        v = fn()
        registrar("ACTIONS_Q5", nome_action, "OK", inicio=t, assinatura=_texto(v, 1000))
except Exception as exc:
    registrar("ACTIONS_Q5", "REAL", "ERRO", inicio=time.perf_counter(), detalhe=str(exc))

## 29. Repartition/coalesce no volume pequeno

In [ ]:
%%spark

try:
    for np in [1, 2, 4, 8, 16, 32]:
        t = time.perf_counter()
        d = df_q5_oficial.repartition(np)
        n = d.count()
        registrar("PARTICOES_SPARK", f"REPARTITION_{np}", "OK", inicio=t,
                  linhas=n, particoes=n_particoes(d))
    for np in [1, 2, 4]:
        t = time.perf_counter()
        d = df_q5_oficial.coalesce(np)
        n = d.count()
        registrar("PARTICOES_SPARK", f"COALESCE_{np}", "OK", inicio=t,
                  linhas=n, particoes=n_particoes(d))
except Exception as exc:
    registrar("PARTICOES_SPARK", "TESTE", "ERRO", inicio=time.perf_counter(), detalhe=str(exc))

## 30. Resultado oficial: microbenchmark 1 linha × 80 colunas

In [ ]:
%%spark

try:
    df80 = spark.range(1).select(
        *[
            (F.lit(CD_CLI).cast("long") if i == 1 else F.lit(i).cast("long")).alias(f"C{i:02d}")
            for i in range(1, 81)
        ]
    )

    operacoes = [
        ("COUNT", lambda: df80.count()),
        ("FIRST", lambda: df80.first()),
        ("COLLECT", lambda: df80.collect()),
    ]
    for nome, fn in operacoes:
        for rep in range(1, 4):
            t = time.perf_counter()
            v = fn()
            registrar("RESULTADO_1X80", nome, "OK", repeticao=rep, inicio=t, assinatura=_texto(v, 700))

    t = time.perf_counter()
    df80.createOrReplaceTempView("vw_lab_1x80")
    registrar("RESULTADO_1X80", "CREATE_TEMP_VIEW", "OK", inicio=t)

    for rep in range(1, 4):
        t = time.perf_counter()
        v = spark.sql("SELECT * FROM vw_lab_1x80").first()
        registrar("RESULTADO_1X80", "SQL_VIEW_FIRST", "OK", repeticao=rep, inicio=t, assinatura=v.asDict())

    try:
        spark.catalog.dropTempView("vw_lab_1x80")
    except Exception:
        pass
except Exception as exc:
    registrar("RESULTADO_1X80", "TESTE", "ERRO", inicio=time.perf_counter(),
              detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}")

## 31. Join/broadcast no Spark — microbenchmark controlado

In [ ]:
%%spark

try:
    fato = spark.range(0, 500000).select(
        "id",
        (F.col("id") % 50).alias("k"),
        (F.col("id") * 0.1).alias("v"),
    )
    dim = spark.range(0, 50).select(F.col("id").alias("k"), F.concat(F.lit("K"), F.col("id")).alias("nome"))

    for estrategia, rhs in [
        ("SEM_HINT", dim),
        ("BROADCAST_EXPLICITO", F.broadcast(dim)),
    ]:
        t = time.perf_counter()
        j = fato.join(rhs, "k", "left")
        n = j.count()
        registrar("JOIN_BROADCAST", estrategia, "OK", inicio=t, linhas=n,
                  particoes=n_particoes(j), assinatura=plano(j, 5000))
except Exception as exc:
    registrar("JOIN_BROADCAST", "TESTE", "ERRO", inicio=time.perf_counter(), detalhe=str(exc))

## 32. Plano físico e AQE — snapshots

In [ ]:
%%spark

planos = {}
for nome, obj in [
    ("Q1", globals().get("df_q1")),
    ("Q3", globals().get("df_q3")),
    ("Q5", globals().get("df_q5_oficial")),
]:
    if obj is not None:
        planos[nome] = plano(obj, 15000)

registrar("PLANOS", "SNAPSHOT", "OK", inicio=time.perf_counter(), assinatura=planos)
CONTEXTO["PLANOS"] = planos

## 33. Payload remoto para benchmark BBMagic

In [ ]:
%%spark

def payload_ascii(n):
    if n <= 0:
        return ""
    prefixo = "<section>"
    sufixo = "</section>"
    corpo = max(0, n - len(prefixo) - len(sufixo))
    s = prefixo + ("x" * corpo) + sufixo
    # ASCII: chars == bytes.
    if len(s) < n:
        s += "x" * (n - len(s))
    return s[:n]

TAMANHOS_PAYLOAD = [
    1024,
    10 * 1024,
    50 * 1024,
    123210,
    250 * 1024,
    500 * 1024,
    1024 * 1024,
]
if EXECUTAR_BENCHMARK_TRANSFERENCIA_GRANDE:
    TAMANHOS_PAYLOAD.append(2 * 1024 * 1024)

META_PAYLOADS = {}
for n in TAMANHOS_PAYLOAD:
    nome = f"PAYLOAD_{n}"
    valor = payload_ascii(n)
    globals()[nome] = valor
    META_PAYLOADS[nome] = {
        "bytes": len(valor.encode("utf-8")),
        "sha256": hashlib.sha256(valor.encode("utf-8")).hexdigest(),
    }

registrar("DASHBOARD", "GERAR_PAYLOADS", "OK", inicio=time.perf_counter(), assinatura=META_PAYLOADS)

## 34. Transferência BBMagic Spark → kernel local

In [ ]:
import time
import hashlib
import statistics

TRANSFERENCIAS_LOCAL = []

try:
    meta = spark.get_from_spark("META_PAYLOADS")
except Exception as exc:
    meta = {}
    print("[LAB][TRANSFER_META_ERRO]", type(exc).__name__, exc)

for nome, esperado in meta.items():
    tempos = []
    oks = []
    for rep in range(1, 4):
        t = time.perf_counter()
        try:
            valor = spark.get_from_spark(nome)
            dur = time.perf_counter() - t
            sha = hashlib.sha256(valor.encode("utf-8")).hexdigest()
            ok = len(valor.encode("utf-8")) == esperado["bytes"] and sha == esperado["sha256"]
            tempos.append(dur)
            oks.append(ok)
            print(f"[LAB][TRANSFER] {nome} rep={rep} {dur:.4f}s checksum={ok}")
        except Exception as exc:
            dur = time.perf_counter() - t
            tempos.append(dur)
            oks.append(False)
            print(f"[LAB][TRANSFER_ERRO] {nome} rep={rep}: {type(exc).__name__}: {exc}")
    TRANSFERENCIAS_LOCAL.append({
        "nome": nome,
        "bytes": esperado["bytes"],
        "tempos_s": tempos,
        "mediana_s": statistics.median(tempos) if tempos else None,
        "checksum_todos_ok": all(oks),
    })

TRANSFERENCIAS_LOCAL

## 35. Geração local de HTML 123210 bytes — custo puro de string

In [ ]:
import time
import hashlib

for rep in range(1, 6):
    t = time.perf_counter()
    prefixo = "<section>"
    sufixo = "</section>"
    corpo = "x" * (123210 - len(prefixo) - len(sufixo))
    html_local_123k = prefixo + corpo + sufixo
    sha = hashlib.sha256(html_local_123k.encode("utf-8")).hexdigest()
    print(f"[LAB][HTML_LOCAL] rep={rep} {time.perf_counter()-t:.6f}s bytes={len(html_local_123k.encode())} sha={sha[:12]}")

## 36. Análise estática opcional dos notebooks Radar presentes no projeto

In [ ]:
import json
import re
from pathlib import Path
import time

PADROES = {
    "count": r"\.count\s*\(",
    "collect": r"\.collect\s*\(",
    "first": r"\.first\s*\(",
    "take": r"\.take\s*\(",
    "show": r"\.show\s*\(",
    "persist": r"\.persist\s*\(",
    "cache": r"\.cache\s*\(",
    "groupBy": r"\.groupBy\s*\(",
    "orderBy": r"\.orderBy\s*\(",
    "join": r"\.join\s*\(",
    "Window": r"\bWindow\b",
    "tempView": r"createOrReplaceTempView",
    "get_from_spark": r"get_from_spark",
}

candidatos = []
for base in [Path.cwd(), Path.cwd()/"notebooks", Path.cwd()/"src"]:
    if base.exists():
        candidatos.extend([p for p in base.glob("*.ipynb") if "radar" in p.name.lower()])

vistos = set()
for p in candidatos:
    if p.resolve() in vistos:
        continue
    vistos.add(p.resolve())
    try:
        nbx = json.loads(p.read_text(encoding="utf-8"))
        totais = {k: 0 for k in PADROES}
        ranking = []
        for i, cell in enumerate(nbx.get("cells", [])):
            src = "".join(cell.get("source", []))
            cs = {k: len(re.findall(rx, src)) for k, rx in PADROES.items()}
            score = sum(cs.values())
            for k, v in cs.items():
                totais[k] += v
            if score:
                ranking.append((score, i, cs, src[:180].replace("\n", " ")))
        print("\n[LAB][STATIC]", p)
        print("totais:", totais)
        for item in sorted(ranking, reverse=True)[:15]:
            print("  ", item)
    except Exception as exc:
        print("[LAB][STATIC_ERRO]", p, type(exc).__name__, exc)

## 37. Ranking remoto e diagnóstico automático

In [ ]:
%%spark

def mediana(vals):
    vals = [float(v) for v in vals if v is not None]
    return statistics.median(vals) if vals else None

def consolidar_equivalencia(rs_ok):
    """
    Retorna estado explícito de equivalência.

    NAO_MEDIDO:
        nenhuma execução OK mediu EQUIVALENTE_BASELINE.

    EQUIVALENTE:
        todas as medições de equivalência existentes foram True.

    DIVERGENTE:
        pelo menos uma medição foi False.

    PARCIAL:
        existe medição de equivalência, mas nem todas as execuções OK
        possuíam medição. Não deve ser promovido automaticamente.
    """
    if not rs_ok:
        return {
            "EQUIVALENTE": None,
            "EQUIVALENCIA_STATUS": "SEM_EXECUCAO_OK",
            "QT_EQ_MEDIDOS": 0,
            "QT_EQ_TRUE": 0,
            "QT_EQ_FALSE": 0,
        }

    medidos = [
        r["EQUIVALENTE_BASELINE"]
        for r in rs_ok
        if r["EQUIVALENTE_BASELINE"] is not None
    ]

    qt_true = sum(v is True for v in medidos)
    qt_false = sum(v is False for v in medidos)

    if not medidos:
        return {
            "EQUIVALENTE": None,
            "EQUIVALENCIA_STATUS": "NAO_MEDIDO",
            "QT_EQ_MEDIDOS": 0,
            "QT_EQ_TRUE": 0,
            "QT_EQ_FALSE": 0,
        }

    if qt_false > 0:
        return {
            "EQUIVALENTE": False,
            "EQUIVALENCIA_STATUS": "DIVERGENTE",
            "QT_EQ_MEDIDOS": len(medidos),
            "QT_EQ_TRUE": qt_true,
            "QT_EQ_FALSE": qt_false,
        }

    if len(medidos) < len(rs_ok):
        return {
            "EQUIVALENTE": None,
            "EQUIVALENCIA_STATUS": "PARCIAL",
            "QT_EQ_MEDIDOS": len(medidos),
            "QT_EQ_TRUE": qt_true,
            "QT_EQ_FALSE": qt_false,
        }

    return {
        "EQUIVALENTE": True,
        "EQUIVALENCIA_STATUS": "EQUIVALENTE",
        "QT_EQ_MEDIDOS": len(medidos),
        "QT_EQ_TRUE": qt_true,
        "QT_EQ_FALSE": qt_false,
    }


# -------------------------------------------------------------------------
# Consolidação por TESTE / ESTRATÉGIA
# -------------------------------------------------------------------------

grupos = {}
for r in RESULTADOS:
    chave = (r["GRUPO"], r["TESTE"], r["ESTRATEGIA"])
    grupos.setdefault(chave, []).append(r)

CONSOLIDADO = []

for (grupo, teste, estrategia), rs in grupos.items():
    ok = [r for r in rs if r["STATUS"] == "OK"]
    eq = consolidar_equivalencia(ok)

    CONSOLIDADO.append({
        "GRUPO": grupo,
        "TESTE": teste,
        "ESTRATEGIA": estrategia,

        "EXECUCOES": len(rs),
        "OK": len(ok),
        "ERROS": sum(r["STATUS"] == "ERRO" for r in rs),
        "WARNS": sum(r["STATUS"] == "WARN" for r in rs),
        "SKIPS": sum(r["STATUS"] == "SKIP" for r in rs),

        "TEMPO_TOTAL_MEDIANA_S": mediana(
            [r["TEMPO_TOTAL_S"] for r in ok]
        ),
        "PREPARACAO_MEDIANA_S": mediana(
            [r["TEMPO_PREPARACAO_S"] for r in ok]
        ),
        "ACTION_MEDIANA_S": mediana(
            [r["TEMPO_ACTION_S"] for r in ok]
        ),

        **eq,
    })


# -------------------------------------------------------------------------
# Ranking geral — gargalos primeiro
# -------------------------------------------------------------------------

print("\n" + "=" * 130)
print("TOP 80 TESTES MAIS LENTOS — TEMPO MEDIANO")
print("=" * 130)

ranking_lentos = sorted(
    [
        x for x in CONSOLIDADO
        if x["TEMPO_TOTAL_MEDIANA_S"] is not None
    ],
    key=lambda z: z["TEMPO_TOTAL_MEDIANA_S"],
    reverse=True,
)

for x in ranking_lentos[:80]:
    print(
        f'{x["TEMPO_TOTAL_MEDIANA_S"]:10.3f}s | '
        f'{x["GRUPO"][:20]:20s} | '
        f'{x["TESTE"][:38]:38s} | '
        f'eq={x["EQUIVALENCIA_STATUS"][:12]:12s} | '
        f'{x["ESTRATEGIA"][:45]}'
    )


# -------------------------------------------------------------------------
# Ranking geral — mais rápidos
# -------------------------------------------------------------------------

print("\n" + "=" * 130)
print("TOP 40 TESTES MAIS RÁPIDOS — TEMPO MEDIANO")
print("=" * 130)

for x in reversed(ranking_lentos[-40:]):
    print(
        f'{x["TEMPO_TOTAL_MEDIANA_S"]:10.3f}s | '
        f'{x["GRUPO"][:20]:20s} | '
        f'{x["TESTE"][:38]:38s} | '
        f'eq={x["EQUIVALENCIA_STATUS"][:12]:12s} | '
        f'{x["ESTRATEGIA"][:45]}'
    )


# -------------------------------------------------------------------------
# Q4 — separar:
# 1. variantes funcionais candidatas;
# 2. diagnósticos físicos/JDBC.
# -------------------------------------------------------------------------

q4_funcionais = [
    x for x in CONSOLIDADO
    if x["GRUPO"] in {
        "Q4_VARIANTES",
        "Q4_BUSCA_REGRESSIVA",
    }
]

q4_diagnosticos = [
    x for x in CONSOLIDADO
    if x["GRUPO"] == "Q4_JDBC_PART"
]

print("\n" + "=" * 130)
print("Q4 — VARIANTES FUNCIONAIS")
print("=" * 130)

for x in sorted(
    q4_funcionais,
    key=lambda z: (
        float("inf")
        if z["TEMPO_TOTAL_MEDIANA_S"] is None
        else z["TEMPO_TOTAL_MEDIANA_S"]
    )
):
    print(x)

print("\n" + "=" * 130)
print("Q4 — DIAGNÓSTICOS FÍSICOS / JDBC (NÃO SÃO CANDIDATOS FUNCIONAIS)")
print("=" * 130)

for x in sorted(
    q4_diagnosticos,
    key=lambda z: (
        float("inf")
        if z["TEMPO_TOTAL_MEDIANA_S"] is None
        else z["TEMPO_TOTAL_MEDIANA_S"]
    )
):
    print(x)


# -------------------------------------------------------------------------
# Erros / warnings / skips
# -------------------------------------------------------------------------

print("\n" + "=" * 130)
print("ERROS / VARIANTES NÃO SUPORTADAS")
print("=" * 130)

for r in RESULTADOS:
    if r["STATUS"] == "ERRO":
        print(
            f'{r["GRUPO"]} | '
            f'{r["TESTE"]} | '
            f'{r["DETALHE"][:600]}'
        )

print("\n" + "=" * 130)
print("WARNINGS")
print("=" * 130)

for r in RESULTADOS:
    if r["STATUS"] == "WARN":
        print(
            f'{r["GRUPO"]} | '
            f'{r["TESTE"]} | '
            f'{r["DETALHE"][:600]}'
        )

print("\n" + "=" * 130)
print("SKIPS POR GATE DE RISCO / PRÉ-CONDIÇÃO")
print("=" * 130)

for r in RESULTADOS:
    if r["STATUS"] == "SKIP":
        print(
            f'{r["GRUPO"]} | '
            f'{r["TESTE"]} | '
            f'{r["DETALHE"][:400]}'
        )


# -------------------------------------------------------------------------
# Recomendações automáticas
# -------------------------------------------------------------------------

RECOMENDACOES = []

def rec(prioridade, tema, evidencia, acao):
    RECOMENDACOES.append({
        "PRIORIDADE": prioridade,
        "TEMA": tema,
        "EVIDENCIA": _texto(evidencia, 4000),
        "ACAO": _texto(acao, 4000),
    })


# -------------------------------------------------------------------------
# Q4: somente estratégia funcional com equivalência EXPLICITAMENTE medida.
# -------------------------------------------------------------------------

q4_eq = [
    x for x in q4_funcionais
    if (
        x["TEMPO_TOTAL_MEDIANA_S"] is not None
        and x["EQUIVALENCIA_STATUS"] == "EQUIVALENTE"
        and x["QT_EQ_MEDIDOS"] > 0
    )
]

if q4_eq:
    melhor = min(
        q4_eq,
        key=lambda x: x["TEMPO_TOTAL_MEDIANA_S"]
    )

    rec(
        "P0/P1",
        "Q4",
        melhor,
        (
            "Candidata técnica somente para homologação adicional. "
            "A equivalência foi efetivamente medida neste cliente. "
            "Repetir contra múltiplos clientes, competências e casos de empate "
            "antes de alterar a baseline."
        ),
    )
else:
    rec(
        "P1",
        "Q4",
        {
            "variantes_funcionais": len(q4_funcionais),
            "equivalentes_comprovadas": 0,
        },
        (
            "Nenhuma variante Q4 possui equivalência explicitamente comprovada "
            "neste passe. Não promover estratégia por tempo apenas."
        ),
    )


# -------------------------------------------------------------------------
# Resultado oficial 1x80.
# -------------------------------------------------------------------------

r80 = [
    x for x in CONSOLIDADO
    if (
        x["GRUPO"] == "RESULTADO_1X80"
        and x["TEMPO_TOTAL_MEDIANA_S"] is not None
    )
]

if r80:
    maior = max(
        x["TEMPO_TOTAL_MEDIANA_S"]
        for x in r80
    )

    if maior < 10:
        rec(
            "P1",
            "Resultado oficial / recomputação",
            {
                "maior_microbench_1x80_s": maior,
                "baseline_resultado_oficial_s":
                    BASELINE_OBSERVADA["RESULTADO_OFICIAL_S"],
            },
            (
                "139 s não são explicados pelo tamanho 1x80. "
                "Instrumentar a lineage e cada action real da etapa de resultado."
            ),
        )


# -------------------------------------------------------------------------
# Scheduler Spark.
# -------------------------------------------------------------------------

sched = [
    x["TEMPO_TOTAL_MEDIANA_S"]
    for x in CONSOLIDADO
    if (
        x["GRUPO"] == "SPARK_SCHEDULER"
        and x["TEMPO_TOTAL_MEDIANA_S"] is not None
    )
]

if sched:
    overhead_action = mediana(sched)

    rec(
        "P1",
        "Overhead por action Spark",
        {
            "range1_count_mediana_s": overhead_action,
        },
        (
            "Usar este valor como custo-base aproximado por action. "
            "Auditorias com muitas actions pequenas podem acumular "
            "tempo significativo."
        ),
    )


# -------------------------------------------------------------------------
# Q5.
# -------------------------------------------------------------------------

if CONTEXTO.get("Q5_QT_CTX") is not None:
    rec(
        "P2",
        "Q5",
        {
            "contexto": CONTEXTO.get("Q5_QT_CTX"),
            "oficial": CONTEXTO.get("Q5_QT_OFICIAL"),
        },
        (
            "Se o volume continuar pequeno e os tempos forem baixos, "
            "não priorizar tuning de Q5; preservar o halo funcional."
        ),
    )


# -------------------------------------------------------------------------
# Resumo de equivalência
# -------------------------------------------------------------------------

print("\n" + "=" * 130)
print("RESUMO DE EQUIVALÊNCIA")
print("=" * 130)

for status in [
    "EQUIVALENTE",
    "DIVERGENTE",
    "PARCIAL",
    "NAO_MEDIDO",
    "SEM_EXECUCAO_OK",
]:
    itens = [
        x for x in CONSOLIDADO
        if x["EQUIVALENCIA_STATUS"] == status
    ]
    print(f"{status:16s}: {len(itens)}")


print("\n" + "=" * 130)
print("RECOMENDAÇÕES AUTOMÁTICAS")
print("=" * 130)

for x in RECOMENDACOES:
    print(f'[{x["PRIORIDADE"]}] {x["TEMA"]}')
    print(" evidência:", x["EVIDENCIA"])
    print(" ação:", x["ACAO"])


RELATORIO_LAB_JSON = json.dumps({
    "contexto": {
        k: v
        for k, v in CONTEXTO.items()
        if k != "PLANOS"
    },
    "resultados": RESULTADOS,
    "consolidado": CONSOLIDADO,
    "recomendacoes": RECOMENDACOES,
}, ensure_ascii=False, default=str)

## 38. Coleta local do relatório e inclusão do benchmark de transferência

In [ ]:
import json
from pathlib import Path
import time

try:
    relatorio_lab = json.loads(spark.get_from_spark("RELATORIO_LAB_JSON"))
    relatorio_lab["transferencias_local"] = TRANSFERENCIAS_LOCAL
    print("[LAB] Resultados remotos:", len(relatorio_lab.get("resultados", [])))
    print("[LAB] Consolidado:", len(relatorio_lab.get("consolidado", [])))
    print("[LAB] Recomendações:", len(relatorio_lab.get("recomendacoes", [])))
except Exception as exc:
    relatorio_lab = {"erro_coleta": f"{type(exc).__name__}: {exc}", "transferencias_local": TRANSFERENCIAS_LOCAL}
    print("[LAB][ERRO_COLETA]", type(exc).__name__, exc)

# Escrita apenas LOCAL no workspace do notebook, nunca em DB2/Hive/HDFS.
try:
    caminho_relatorio = Path.cwd() / f"radar_laboratorio_{468459778}_resultado.json"
    caminho_relatorio.write_text(
        json.dumps(relatorio_lab, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    print("[LAB] Relatório local salvo em:", caminho_relatorio)
except Exception as exc:
    print("[LAB][WARN] Não foi possível salvar relatório local:", type(exc).__name__, exc)

## 39. Limpeza de caches criados pelo laboratório

In [ ]:
%%spark

inicio = time.perf_counter()
erros = []
removidos = 0

for d in list(PERSISTIDOS):
    try:
        d.unpersist(blocking=False)
        removidos += 1
    except Exception as exc:
        erros.append(str(exc))

for nome in list(globals()):
    if nome.startswith("PAYLOAD_"):
        try:
            globals().pop(nome, None)
        except Exception:
            pass

registrar(
    "LIMPEZA", "UNPERSIST_E_PAYLOADS",
    "OK" if not erros else "WARN",
    inicio=inicio,
    assinatura={"unpersist": removidos, "erros": erros},
)

# O que enviar depois da execução

Para uma análise completa, envie o arquivo:

`radar_laboratorio_468459778_resultado.json`

Se preferir colar somente partes, envie:

1. **Q4 — VARIANTES FUNCIONAIS** e **Q4 — DIAGNÓSTICOS FÍSICOS / JDBC**
2. **ERROS / VARIANTES NÃO SUPORTADAS**
3. **RECOMENDAÇÕES AUTOMÁTICAS**
4. resultados `SPARK_SCHEDULER`
5. resultados `RESULTADO_1X80`
6. `TRANSFERENCIAS_LOCAL`
7. `Q1_FETCHSIZE`
8. `Q5_FETCHSIZE`
9. `Q5_JDBC_PART`
10. `Q4_JDBC_PART`
11. `CACHE_LINEAGE`
12. `ACTIONS_Q5`

## Como interpretar

- Variante Q4 muito mais rápida **e equivalente** → candidata a homologação adicional.
- Variante rápida, mas não equivalente → descartar como otimização funcional.
- CTE falhar → evidência real do wrapper atual, não mais apenas hipótese documental.
- 1×80 rápido → o gargalo de 139 s está antes da montagem da linha.
- 123210 bytes transferirem rápido → os 126 s do dashboard não são HTML/BBMagic.
- `range(1).count()` caro → cada action Spark tem overhead estrutural relevante.
- segundo `count` em cache muito mais rápido → cache ajuda somente onde há reuso.
- `numPartitions > 1` piorar Q5 pequeno → paralelismo JDBC tem overhead maior que o benefício nesse recorte.
- função/cast na chave piorar muito → evidência prática de perda do caminho de acesso.

# Execução recomendada por fases

### Fase 1 — agora

Mantenha:

```python
MODO_LAB = "SEGURO"
```

Execute o notebook inteiro.

### Fase 2 — somente depois de analisar o JSON da Fase 1

Se houver evidência de que precisamos aprofundar DB2:

```python
MODO_LAB = "AGRESSIVO"
```

Execute novamente. Os testes `EXTREMO` ainda permanecem bloqueados.

### Fase 3 — excepcional

Somente para provar comportamento deliberadamente ruim:

```python
MODO_LAB = "EXTREMO"
```

Essa fase inclui funções na chave das fontes gigantes e outras formulações com risco real de caminho de acesso ruim.

**Não é necessário chegar à Fase 3 para otimizar o Radar.**